<a href="https://colab.research.google.com/github/tffnnnnnn/Machine_Learning_MASnI3/blob/main/%E6%90%9C%E5%B0%8BNOMAD_MASnI3%E7%9A%84DFT%E8%B3%87%E6%96%99.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**確認API是否正常運作，且可讀取622筆公開資料**

In [ ]:

import requests

# NOMAD 官方 API
API_URL = "https://nomad-lab.eu/prod/v1/api/v1"

# MASnI3 化學式：CH6I3NSn
query = {
    "results.material.chemical_formula_hill": "CH6I3NSn",
    "results.method.method_name": "DFT"
}

# 先確認搜尋數量與回傳格式
payload = {
    "owner": "public",
    "query": query,
    "pagination": {
        "page_size": 5
    }
}

response = requests.post(
    f"{API_URL}/entries/query",
    json=payload,
    timeout=60
)

response.raise_for_status()
result = response.json()

print("=== NOMAD MASnI3 DFT Search ===")
print("Total matching entries:", result["pagination"]["total"])
print("Returned entries:", len(result["data"]))

print("\n=== First 5 Entries ===")

for i, entry in enumerate(result["data"], 1):
    print(f"\nEntry {i}")
    print("Entry ID:", entry.get("entry_id"))

    material = entry.get("results", {}).get("material", {})
    print("Formula:", material.get("chemical_formula_hill"))

    method = entry.get("results", {}).get("method", {})
    print("Method:", method.get("method_name"))


=== NOMAD MASnI3 DFT Search ===
Total matching entries: 622
Returned entries: 5

=== First 5 Entries ===

Entry 1
Entry ID: --aVj0_fQXVFQuzQ7FWUL8pxX_C8
Formula: CH6I3NSn
Method: DFT

Entry 2
Entry ID: -SCdLmiM-XK_5m6JvmNckuodlpLa
Formula: CH6I3NSn
Method: DFT

Entry 3
Entry ID: -areUk8CdbRLUJSgEGt6VK116A3L
Formula: CH6I3NSn
Method: DFT

Entry 4
Entry ID: -bc76c6qQnD_laqNST6-8DVJkY8y
Formula: CH6I3NSn
Method: DFT

Entry 5
Entry ID: -i5blgHtFqEoLVzuNZtqcyl4rTHr
Formula: CH6I3NSn
Method: DFT


In [ ]:
import requests

all_entries = []
page_after = None

while True:
    payload = {
        "owner": "public",
        "query": query,
        "pagination": {
            "page_size": 100
        },
        "required": {
            "include": [
                "entry_id",
                "upload_id",
                "results.material.chemical_formula_hill",
                "results.method.method_name"
            ]
        }
    }

    if page_after is not None:
        payload["pagination"]["page_after_value"] = page_after

    response = requests.post(
        f"{API_URL}/entries/query",
        json=payload,
        timeout=60
    )

    response.raise_for_status()
    result = response.json()

    all_entries.extend(result["data"])

    print(f"Retrieved: {len(all_entries)} entries")

    page_after = result["pagination"].get(
        "next_page_after_value"
    )

    if not page_after:
        break

# 確認搜尋結果
entry_ids = [entry["entry_id"] for entry in all_entries]

print("\n=== NOMAD Dataset Verification ===")
print("Total retrieved:", len(all_entries))
print("Unique Entry IDs:", len(set(entry_ids)))
print("Expected total:", result["pagination"]["total"])

if len(all_entries) == result["pagination"]["total"]:
    print("\nAll MASnI3 DFT entries retrieved successfully!")
else:
    print("\nWarning: Retrieved count differs from expected total.")

Retrieved: 100 entries
Retrieved: 200 entries
Retrieved: 300 entries
Retrieved: 400 entries
Retrieved: 500 entries
Retrieved: 600 entries
Retrieved: 622 entries

=== NOMAD Dataset Verification ===
Total retrieved: 622
Unique Entry IDs: 622
Expected total: 622

All MASnI3 DFT entries retrieved successfully!


**檢查 DFT Archive資料格式**

In [ ]:
import requests

# 取得第一筆 MASnI3 DFT 計算的 Archive
test_entry_id = entry_ids[0]

url = f"{API_URL}/entries/{test_entry_id}/archive"

response = requests.get(url, timeout=60)
response.raise_for_status()

archive_response = response.json()
archive = archive_response.get("data", archive_response)

print("=== NOMAD DFT Archive Inspection ===")
print("Entry ID:", test_entry_id)

print("\n=== Top-level Keys ===")
print(list(archive.keys()))

print("\n=== Run Information ===")

runs = archive.get("run", [])

print("Number of runs:", len(runs))

if runs:
    run = runs[0]

    print("Run keys:", list(run.keys()))
    print("Number of systems:", len(run.get("system", [])))
    print("Number of calculations:", len(run.get("calculation", [])))

    calculations = run.get("calculation", [])

    if calculations:
        calc = calculations[-1]

        print("\n=== Calculation Keys ===")
        print(list(calc.keys()))

        print("\n=== DFT Data Availability ===")
        print("Energy:", "energy" in calc)
        print("Forces:", "forces" in calc)
        print("Stress:", "stress" in calc)

else:
    print("No run section found.")
    print("We need to inspect the archive structure.")

=== NOMAD DFT Archive Inspection ===
Entry ID: --aVj0_fQXVFQuzQ7FWUL8pxX_C8

=== Top-level Keys ===
['entry_id', 'upload_id', 'parser_name', 'archive']

=== Run Information ===
Number of runs: 0
No run section found.
We need to inspect the archive structure.


**讀取一筆資料**

In [ ]:
# 修正 NOMAD Archive 資料層級
archive = archive_response["data"]["archive"]

print("=== Corrected NOMAD Archive Inspection ===")

print("\nArchive keys:")
print(list(archive.keys()))

runs = archive.get("run", [])

print("\nNumber of runs:", len(runs))

if runs:
    run = runs[0]

    print("\nRun keys:")
    print(list(run.keys()))

    print("\nNumber of systems:", len(run.get("system", [])))
    print("Number of calculations:", len(run.get("calculation", [])))

    calculations = run.get("calculation", [])

    if calculations:
        calc = calculations[-1]

        print("\n=== DFT Data Availability ===")

        for key in ["energy", "forces", "stress"]:
            value = calc.get(key)

            print(f"{key}: {'Available' if value is not None else 'Not found'}")

        print("\nCalculation keys:")
        print(list(calc.keys()))
else:
    print("No run section found in archive.")

=== Corrected NOMAD Archive Inspection ===

Archive keys:
['processing_logs', 'run', 'workflow', 'workflow2', 'metadata', 'results', 'm_ref_archives']

Number of runs: 1

Run keys:
['x_fhi_aims_number_of_tasks', 'x_fhi_aims_program_compilation_date', 'x_fhi_aims_program_compilation_time', 'x_fhi_aims_program_execution_date', 'x_fhi_aims_program_execution_time', 'x_fhi_aims_section_parallel_tasks', 'program', 'time_run', 'method', 'system', 'calculation']

Number of systems: 1
Number of calculations: 1

=== DFT Data Availability ===
energy: Available
forces: Not found
stress: Not found

Calculation keys:
['system_ref', 'method_ref', 'calculation_converged', 'x_fhi_aims_energy_C_LDA', 'x_fhi_aims_energy_X_LDA', 'x_fhi_aims_calculation_md', 'n_scf_iterations', 'energy', 'eigenvalues', 'scf_iteration']


**讀取662筆資料**

In [ ]:
import requests
import pandas as pd
import time
from collections import Counter

# =====================================================
# NOMAD MASnI3 DFT Dataset Analysis
# =====================================================

records = []
failed_entries = []

session = requests.Session()

def has_value(section):
    """檢查 NOMAD 物理量是否包含非空數值。"""
    if section is None:
        return False

    if isinstance(section, dict):
        if "total" in section:
            return has_value(section["total"])
        if "value" in section:
            return has_value(section["value"])
        return False

    if isinstance(section, (list, tuple)):
        return len(section) > 0

    return True


for i, entry_id in enumerate(entry_ids, 1):

    try:
        url = f"{API_URL}/entries/{entry_id}/archive"

        response = session.get(url, timeout=60)
        response.raise_for_status()

        result = response.json()
        archive = result["data"]["archive"]

        has_structure = False
        has_energy = False
        has_forces = False
        has_stress = False

        program_names = set()
        calculation_count = 0

        for run in archive.get("run", []):

            program = run.get("program", {})
            program_names.add(program.get("name", "Unknown"))

            for system in run.get("system", []):
                atoms = system.get("atoms", {})

                if (
                    atoms.get("positions") is not None
                    and atoms.get("lattice_vectors") is not None
                    and atoms.get("labels") is not None
                ):
                    has_structure = True

            for calc in run.get("calculation", []):

                calculation_count += 1

                if has_value(calc.get("energy")):
                    has_energy = True

                if has_value(calc.get("forces")):
                    has_forces = True

                if has_value(calc.get("stress")):
                    has_stress = True

        records.append({
            "entry_id": entry_id,
            "program": ", ".join(sorted(program_names)),
            "n_calculations": calculation_count,
            "structure": has_structure,
            "energy": has_energy,
            "forces": has_forces,
            "stress": has_stress,
            "energy_forces": has_energy and has_forces,
            "complete_EFS": (
                has_structure
                and has_energy
                and has_forces
                and has_stress
            )
        })

    except Exception as e:
        failed_entries.append({
            "entry_id": entry_id,
            "error": str(e)
        })

    # 每50筆顯示進度並保存
    if i % 50 == 0 or i == len(entry_ids):
        print(f"Processed: {i}/{len(entry_ids)}")

        pd.DataFrame(records).to_csv(
            "MASnI3_DFT_availability.csv",
            index=False
        )

    # 避免短時間向 NOMAD 發送過多請求
    time.sleep(0.2)


# =====================================================
# Summary
# =====================================================

df = pd.DataFrame(records)

print("\n=== MASnI3 DFT Dataset Summary ===")

print("Total searched:", len(entry_ids))
print("Successfully analyzed:", len(df))
print("Failed entries:", len(failed_entries))

print("\n=== Data Availability ===")

for column in [
    "structure",
    "energy",
    "forces",
    "stress",
    "energy_forces",
    "complete_EFS"
]:
    print(f"{column}: {int(df[column].sum())}")

print("\n=== Calculation Programs ===")
print(df["program"].value_counts())

print("\n=== Calculation Count ===")
print("Total calculations:", int(df["n_calculations"].sum()))

print("\nResults saved:")
print("MASnI3_DFT_availability.csv")

if failed_entries:
    pd.DataFrame(failed_entries).to_csv(
        "MASnI3_failed_entries.csv",
        index=False
    )
    print("Failed entries saved: MASnI3_failed_entries.csv")

**確認201筆無法讀取原因**

In [ ]:
import pandas as pd
from collections import Counter

# =====================================================
# Step 5: Analyze Failed NOMAD Entries
# =====================================================

failed_df = pd.DataFrame(failed_entries)

print("=== Failed Entries Analysis ===")
print("Total failed entries:", len(failed_df))

# 依錯誤類型分類
def classify_error(error):
    error = str(error)

    if "429" in error:
        return "Rate limit (429)"
    elif "404" in error:
        return "Not found (404)"
    elif "403" in error:
        return "Access denied (403)"
    elif "500" in error:
        return "Server error (500)"
    elif "502" in error:
        return "Bad gateway (502)"
    elif "503" in error:
        return "Service unavailable (503)"
    elif "504" in error:
        return "Gateway timeout (504)"
    elif "timed out" in error.lower():
        return "Timeout"
    elif "connection" in error.lower():
        return "Connection error"
    elif "archive" in error.lower():
        return "Archive error"
    else:
        return "Other error"

failed_df["error_type"] = failed_df["error"].apply(classify_error)

print("\n=== Error Type Distribution ===")
print(failed_df["error_type"].value_counts())

print("\n=== Example Errors ===")

for error_type, group in failed_df.groupby("error_type"):
    print(f"\n[{error_type}]")
    print("Example Entry ID:", group.iloc[0]["entry_id"])
    print("Error message:", group.iloc[0]["error"][:300])

print("\n=== Successful Entries by Program ===")
print(df["program"].value_counts())

print("\n=== Complete EFS Entries by Program ===")
print(
    df[df["complete_EFS"] == True]["program"].value_counts()
)

=== Failed Entries Analysis ===
Total failed entries: 201

=== Error Type Distribution ===
error_type
Rate limit (429)    201
Name: count, dtype: int64

=== Example Errors ===

[Rate limit (429)]
Example Entry ID: 2GygyOufhgnqndE9TbHfDo7RWQcq
Error message: 429 Client Error: Too Many Requests for url: https://nomad-lab.eu/prod/v1/api/v1/entries/2GygyOufhgnqndE9TbHfDo7RWQcq/archive

=== Successful Entries by Program ===
program
FHI-aims            390
VASP                 29
Quantum Espresso      2
Name: count, dtype: int64

=== Complete EFS Entries by Program ===
program
VASP    29
Name: count, dtype: int64


**API恢復正常**

In [ ]:
import requests

# 從先前失敗的201筆中選擇第一筆
test_failed_id = failed_entries[0]["entry_id"]

url = f"{API_URL}/entries/{test_failed_id}/archive"

print("=== NOMAD Rate Limit Recovery Test ===")
print("Testing Entry:", test_failed_id)

response = requests.get(
    url,
    timeout=60
)

print("HTTP Status:", response.status_code)
print("Retry-After:", response.headers.get("Retry-After"))

if response.status_code == 200:

    result = response.json()
    archive = result["data"]["archive"]

    print("\nAPI connection recovered!")
    print("Archive keys:", list(archive.keys()))
    print("Number of runs:", len(archive.get("run", [])))

elif response.status_code == 429:

    print("\nRate limit is still active.")
    print("Do not continue bulk downloading yet.")

else:

    print("\nUnexpected HTTP status.")
    print("Response:", response.text[:300])

=== NOMAD Rate Limit Recovery Test ===
Testing Entry: 2GygyOufhgnqndE9TbHfDo7RWQcq
HTTP Status: 200
Retry-After: None

API connection recovered!
Archive keys: ['processing_logs', 'run', 'workflow', 'workflow2', 'metadata', 'results', 'm_ref_archives']
Number of runs: 1


**下載未成功的201筆**

In [ ]:
import requests
import pandas as pd
import time

# =====================================================
# Step 7: Retry Failed NOMAD Entries
# =====================================================

session = requests.Session()

# 原本已成功讀取的資料
existing_ids = set(df["entry_id"])

# 只處理尚未成功的 Entry
retry_ids = [
    item["entry_id"]
    for item in failed_entries
    if item["entry_id"] not in existing_ids
]

print("=== NOMAD Retry Process ===")
print("Entries to retry:", len(retry_ids))

new_records = []
remaining_failed = []

for i, entry_id in enumerate(retry_ids, 1):

    try:
        url = f"{API_URL}/entries/{entry_id}/archive"

        response = session.get(url, timeout=60)

        # 如果再次遇到限制，停止請求
        if response.status_code == 429:
            print("\nRate limit encountered. Stopping safely.")
            remaining_failed.extend(retry_ids[i-1:])
            break

        response.raise_for_status()

        result = response.json()
        archive = result["data"]["archive"]

        has_structure = False
        has_energy = False
        has_forces = False
        has_stress = False

        program_names = set()
        calculation_count = 0

        for run in archive.get("run", []):

            program_names.add(
                run.get("program", {}).get("name", "Unknown")
            )

            for system in run.get("system", []):

                atoms = system.get("atoms", {})

                if (
                    atoms.get("positions") is not None
                    and atoms.get("lattice_vectors") is not None
                    and atoms.get("labels") is not None
                ):
                    has_structure = True

            for calc in run.get("calculation", []):

                calculation_count += 1

                if has_value(calc.get("energy")):
                    has_energy = True

                if has_value(calc.get("forces")):
                    has_forces = True

                if has_value(calc.get("stress")):
                    has_stress = True

        new_records.append({
            "entry_id": entry_id,
            "program": ", ".join(sorted(program_names)),
            "n_calculations": calculation_count,
            "structure": has_structure,
            "energy": has_energy,
            "forces": has_forces,
            "stress": has_stress,
            "energy_forces": has_energy and has_forces,
            "complete_EFS": (
                has_structure
                and has_energy
                and has_forces
                and has_stress
            )
        })

    except Exception as e:

        remaining_failed.append(entry_id)
        print(f"Failed: {entry_id} | {str(e)[:100]}")

    # 合併並定期保存
    if i % 20 == 0 or i == len(retry_ids):

        combined_df = pd.concat(
            [df, pd.DataFrame(new_records)],
            ignore_index=True
        )

        combined_df = combined_df.drop_duplicates(
            subset="entry_id"
        )

        combined_df.to_csv(
            "MASnI3_DFT_availability_updated.csv",
            index=False
        )

        print(f"Processed: {i}/{len(retry_ids)}")
        print(f"Total successful: {len(combined_df)}")

    # 降低 API 請求頻率
    time.sleep(2)


# =====================================================
# Final Summary
# =====================================================

combined_df = pd.concat(
    [df, pd.DataFrame(new_records)],
    ignore_index=True
)

combined_df = combined_df.drop_duplicates(subset="entry_id")

combined_df.to_csv(
    "MASnI3_DFT_availability_updated.csv",
    index=False
)

print("\n=== Updated MASnI3 DFT Summary ===")

print("Total NOMAD entries:", len(entry_ids))
print("Successfully analyzed:", len(combined_df))
print("Remaining failed:", len(entry_ids) - len(combined_df))

print("\n=== DFT Data Availability ===")

for column in [
    "structure",
    "energy",
    "forces",
    "stress",
    "energy_forces",
    "complete_EFS"
]:
    print(f"{column}: {int(combined_df[column].sum())}")

print("\n=== Calculation Programs ===")
print(combined_df["program"].value_counts())

print("\n=== Complete EFS by Program ===")
print(
    combined_df[combined_df["complete_EFS"] == True]
    ["program"].value_counts()
)

print("\nFile saved:")
print("MASnI3_DFT_availability_updated.csv")

=== NOMAD Retry Process ===
Entries to retry: 201
Processed: 20/201
Total successful: 441
Processed: 40/201
Total successful: 461
Processed: 60/201
Total successful: 481
Processed: 80/201
Total successful: 501
Processed: 100/201
Total successful: 521
Processed: 120/201
Total successful: 541
Processed: 140/201
Total successful: 561
Processed: 160/201
Total successful: 581
Processed: 180/201
Total successful: 601
Processed: 200/201
Total successful: 621
Processed: 201/201
Total successful: 622

=== Updated MASnI3 DFT Summary ===
Total NOMAD entries: 622
Successfully analyzed: 622
Remaining failed: 0

=== DFT Data Availability ===
structure: 622
energy: 622
forces: 41
stress: 45
energy_forces: 41
complete_EFS: 41

=== Calculation Programs ===
program
FHI-aims            577
VASP                 41
Quantum Espresso      4
Name: count, dtype: int64

=== Complete EFS by Program ===
program
VASP    41
Name: count, dtype: int64

File saved:
MASnI3_DFT_availability_updated.csv


**檢查 41 筆完整 EFS資料**

In [ ]:
import requests

# 選取第一筆具有 Energy + Forces + Stress 的 VASP 資料
efs_entries = combined_df[
    combined_df["complete_EFS"] == True
]

test_id = efs_entries.iloc[0]["entry_id"]

response = requests.get(
    f"{API_URL}/entries/{test_id}/archive",
    timeout=60
)

response.raise_for_status()

archive = response.json()["data"]["archive"]

print("=== MASnI3 VASP DFT Data Inspection ===")
print("Entry ID:", test_id)

for run_index, run in enumerate(archive.get("run", [])):

    print(f"\n=== Run {run_index} ===")
    print("Program:", run.get("program", {}).get("name"))

    calculations = run.get("calculation", [])
    systems = run.get("system", [])

    print("Number of calculations:", len(calculations))
    print("Number of systems:", len(systems))

    if calculations:

        calc = calculations[-1]

        for name in ["energy", "forces", "stress"]:

            section = calc.get(name)

            print(f"\n=== {name.upper()} ===")

            if isinstance(section, dict):
                print("Available keys:", list(section.keys()))

                total = section.get("total")

                if isinstance(total, dict):
                    print("Total keys:", list(total.keys()))

                    if "value" in total:
                        value = total["value"]

                        if isinstance(value, list):
                            print("Value length:", len(value))
                            print("First value:", value[0] if value else None)
                        else:
                            print("Value:", value)

                else:
                    print("Total:", total)

            else:
                print("Section:", section)

=== MASnI3 VASP DFT Data Inspection ===
Entry ID: -bc76c6qQnD_laqNST6-8DVJkY8y

=== Run 0 ===
Program: VASP
Number of calculations: 1
Number of systems: 1

=== ENERGY ===
Available keys: ['fermi', 'highest_occupied', 'lowest_unoccupied', 'total', 'free', 'total_t0']
Total keys: ['value']
Value: -6.997181075759813e-18

=== FORCES ===
Available keys: ['total']
Total keys: ['value']
Value length: 12
First value: [9.934456436780398e-13, -3.9107529459305995e-12, -2.0177492093269197e-12]

=== STRESS ===
Available keys: ['total']
Total keys: ['value']
Value length: 3
First value: [1535386.0, 4224196.0, 6503219.0]


**確認資料數值是否合理**

In [ ]:
import numpy as np

# =====================================================
# NOMAD DFT Unit Conversion
# =====================================================

# 物理常數
EV_TO_J = 1.602176634e-19
ANGSTROM_TO_M = 1e-10

# 取得最後一個 DFT 計算結果
calc = archive["run"][0]["calculation"][-1]

# 1. Energy: J → eV
energy_j = calc["energy"]["total"]["value"]
energy_ev = energy_j / EV_TO_J

# 2. Forces: N → eV/Å
forces_n = np.array(
    calc["forces"]["total"]["value"],
    dtype=float
)

forces_ev_ang = forces_n * ANGSTROM_TO_M / EV_TO_J

# 3. Stress: Pa → GPa
stress_pa = np.array(
    calc["stress"]["total"]["value"],
    dtype=float
)

stress_gpa = stress_pa / 1e9

# 4. Number of atoms
systems = archive["run"][0]["system"]
atoms = systems[-1]["atoms"]

n_atoms = len(atoms["labels"])

energy_per_atom = energy_ev / n_atoms

# =====================================================
# Results
# =====================================================

print("=== MASnI3 DFT Data Conversion ===")

print("\nNumber of atoms:", n_atoms)

print("\n=== Energy ===")
print(f"Total Energy: {energy_ev:.6f} eV")
print(f"Energy per atom: {energy_per_atom:.6f} eV/atom")

print("\n=== Forces ===")
print("Shape:", forces_ev_ang.shape)
print("First atomic force:", forces_ev_ang[0])
print(
    "Maximum force magnitude:",
    np.linalg.norm(forces_ev_ang, axis=1).max(),
    "eV/Å"
)

print("\n=== Stress ===")
print("Shape:", stress_gpa.shape)
print("Stress tensor (GPa):")
print(stress_gpa)

print("\n=== Data Validation ===")

print("Energy finite:", np.isfinite(energy_ev))
print("Forces finite:", np.isfinite(forces_ev_ang).all())
print("Stress finite:", np.isfinite(stress_gpa).all())

print(
    "Force shape valid:",
    forces_ev_ang.shape == (n_atoms, 3)
)

print(
    "Stress shape valid:",
    stress_gpa.shape == (3, 3)
)

=== MASnI3 DFT Data Conversion ===

Number of atoms: 12

=== Energy ===
Total Energy: -43.672969 eV
Energy per atom: -3.639414 eV/atom

=== Forces ===
Shape: (12, 3)
First atomic force: [ 0.00062006 -0.0024409  -0.00125938]
Maximum force magnitude: 0.03173838415877374 eV/Å

=== Stress ===
Shape: (3, 3)
Stress tensor (GPa):
[[ 0.00153539  0.0042242   0.00650322]
 [ 0.00422424  0.00052185 -0.00843918]
 [ 0.00650325 -0.00843923 -0.08899009]]

=== Data Validation ===
Energy finite: True
Forces finite: True
Stress finite: True
Force shape valid: True
Stress shape valid: True


In [ ]:
# =====================================================
# Step 10: Verify DFT Structure–Label Association
# =====================================================

run = archive["run"][0]
calc = run["calculation"][-1]

print("=== DFT Structure–Label Association ===")

print("\nCalculation references:")
print("system_ref:", calc.get("system_ref"))
print("method_ref:", calc.get("method_ref"))

print("\n=== Available Systems ===")

for i, system in enumerate(run.get("system", [])):

    atoms = system.get("atoms", {})

    print(f"\nSystem {i}")
    print("System keys:", list(system.keys()))
    print("Atom keys:", list(atoms.keys()))

    labels = atoms.get("labels", [])

    print("Number of atoms:", len(labels))
    print("Atomic species:", sorted(set(labels)))

print("\n=== Calculation Status ===")

print(
    "Calculation converged:",
    calc.get("calculation_converged")
)

print(
    "Number of SCF iterations:",
    calc.get("n_scf_iterations")
)

print("\n=== DFT Method Information ===")

for i, method in enumerate(run.get("method", [])):

    print(f"\nMethod {i}")
    print("Available keys:", list(method.keys()))

    dft = method.get("dft", {})

    print("DFT keys:", list(dft.keys()))

=== DFT Structure–Label Association ===

Calculation references:
system_ref: /run/0/system/0
method_ref: /run/0/method/0

=== Available Systems ===

System 0
System keys: ['type', 'configuration_raw_gid', 'is_representative', 'chemical_composition', 'chemical_composition_hill', 'chemical_composition_reduced', 'atoms', 'symmetry']
Atom keys: ['species', 'labels', 'positions', 'lattice_vectors', 'periodic']
Number of atoms: 12
Atomic species: ['C', 'H', 'I', 'N', 'Sn']

=== Calculation Status ===
Calculation converged: None
Number of SCF iterations: None

=== DFT Method Information ===

Method 0
Available keys: ['dft', 'electronic', 'scf', 'atom_parameters', 'basis_set']
DFT keys: ['xc_functional']


**批次驗證 41 筆 VASP 資料**

In [ ]:
import requests
import numpy as np
import pandas as pd
import time

# =====================================================
# Step 11: Validate 41 VASP DFT Entries
# =====================================================

EV_TO_J = 1.602176634e-19
ANGSTROM_TO_M = 1e-10

vasp_ids = combined_df.loc[
    combined_df["complete_EFS"] == True,
    "entry_id"
].tolist()

validation_records = []
vasp_archives = {}
validation_errors = []

session = requests.Session()

for i, entry_id in enumerate(vasp_ids, 1):

    try:
        response = session.get(
            f"{API_URL}/entries/{entry_id}/archive",
            timeout=60
        )

        if response.status_code == 429:
            print("Rate limit reached. Stopping safely.")
            break

        response.raise_for_status()

        archive_data = response.json()["data"]["archive"]
        vasp_archives[entry_id] = archive_data

        valid_calculations = 0

        for run_index, run in enumerate(archive_data.get("run", [])):

            systems = run.get("system", [])
            methods = run.get("method", [])

            for calc_index, calc in enumerate(
                run.get("calculation", [])
            ):

                # 檢查 Energy、Forces、Stress 是否在同一 Calculation
                try:
                    energy = calc["energy"]["total"]["value"]
                    forces = np.asarray(
                        calc["forces"]["total"]["value"],
                        dtype=float
                    )
                    stress = np.asarray(
                        calc["stress"]["total"]["value"],
                        dtype=float
                    )

                    # 解析 Calculation 對應的 System
                    system_ref = calc.get("system_ref", "")
                    method_ref = calc.get("method_ref", "")

                    expected_ref_prefix = f"/run/{run_index}/"

                    if not system_ref.startswith(
                        expected_ref_prefix + "system/"
                    ):
                        continue

                    system_index = int(system_ref.split("/")[-1])
                    system = systems[system_index]

                    atoms = system["atoms"]
                    labels = atoms["labels"]

                    n_atoms = len(labels)

                    # 驗證晶體結構
                    positions = np.asarray(
                        atoms["positions"],
                        dtype=float
                    )

                    lattice = np.asarray(
                        atoms["lattice_vectors"],
                        dtype=float
                    )

                    # 驗證數值與維度
                    valid = (
                        np.isfinite(energy)
                        and np.isfinite(forces).all()
                        and np.isfinite(stress).all()
                        and positions.shape == (n_atoms, 3)
                        and lattice.shape == (3, 3)
                        and forces.shape == (n_atoms, 3)
                        and stress.shape == (3, 3)
                        and n_atoms > 0
                    )

                    if not valid:
                        continue

                    # 確認化學組成
                    from collections import Counter

                    composition = Counter(labels)

                    expected_composition = {
                        "Sn": 1,
                        "I": 3,
                        "C": 1,
                        "N": 1,
                        "H": 6
                    }

                    # 允許 12 原子晶胞或其整數倍超晶胞
                    formula_units = n_atoms // 12

                    composition_valid = (
                        n_atoms % 12 == 0
                        and all(
                            composition.get(element, 0)
                            == count * formula_units
                            for element, count
                            in expected_composition.items()
                        )
                        and len(composition) == 5
                    )

                    if not composition_valid:
                        continue

                    valid_calculations += 1

                    # 單位轉換
                    energy_ev = energy / EV_TO_J

                    forces_ev_ang = (
                        forces * ANGSTROM_TO_M / EV_TO_J
                    )

                    stress_gpa = stress / 1e9

                    validation_records.append({
                        "entry_id": entry_id,
                        "run_index": run_index,
                        "calc_index": calc_index,
                        "system_ref": system_ref,
                        "method_ref": method_ref,
                        "n_atoms": n_atoms,
                        "energy_eV": energy_ev,
                        "energy_eV_atom": energy_ev / n_atoms,
                        "max_force_eV_A": np.linalg.norm(
                            forces_ev_ang, axis=1
                        ).max(),
                        "max_abs_stress_GPa": np.abs(
                            stress_gpa
                        ).max(),
                        "valid_EFS": True
                    })

                except (
                    KeyError,
                    TypeError,
                    ValueError,
                    IndexError,
                    ZeroDivisionError
                ):
                    continue

    except Exception as e:
        validation_errors.append({
            "entry_id": entry_id,
            "error": str(e)
        })

    if i % 10 == 0 or i == len(vasp_ids):
        print(f"Processed: {i}/{len(vasp_ids)}")

    time.sleep(2)


# =====================================================
# Summary
# =====================================================

validated_df = pd.DataFrame(validation_records)

print("\n=== MASnI3 VASP DFT Validation ===")
print("Candidate Entries:", len(vasp_ids))
print("Successfully retrieved:", len(vasp_archives))
print("Retrieval errors:", len(validation_errors))

if not validated_df.empty:

    print(
        "Entries with valid EFS:",
        validated_df["entry_id"].nunique()
    )

    print(
        "Valid DFT configurations:",
        len(validated_df)
    )

    print("\n=== Atomic Count Distribution ===")
    print(validated_df["n_atoms"].value_counts())

    print("\n=== Energy Range (eV/atom) ===")
    print(
        validated_df["energy_eV_atom"].describe()
    )

    validated_df.to_csv(
        "MASnI3_VASP_validated_EFS.csv",
        index=False
    )

    print("\nSaved: MASnI3_VASP_validated_EFS.csv")

else:
    print("No fully validated EFS configurations found.")

Processed: 10/41
Processed: 20/41
Processed: 30/41
Processed: 40/41
Processed: 41/41

=== MASnI3 VASP DFT Validation ===
Candidate Entries: 41
Successfully retrieved: 41
Retrieval errors: 0
Entries with valid EFS: 34
Valid DFT configurations: 55

=== Atomic Count Distribution ===
n_atoms
12    55
Name: count, dtype: int64

=== Energy Range (eV/atom) ===
count    55.000000
mean     -4.373081
std       1.059491
min      -5.991275
25%      -5.987722
50%      -3.641353
75%      -3.640542
max      -3.594969
Name: energy_eV_atom, dtype: float64

Saved: MASnI3_VASP_validated_EFS.csv


**能量至少呈現兩個明顯區間**：
- 約 −5.99 eV/atom
- 約 −3.64 eV/atom


需確認DFT計算方式是否相同

In [ ]:
import pandas as pd
import json

# =====================================================
# Step 12: Analyze DFT Energy Groups and Methods
# =====================================================

method_records = []

for _, row in validated_df.iterrows():

    entry_id = row["entry_id"]
    run_index = int(row["run_index"])
    method_ref = row["method_ref"]

    archive_data = vasp_archives[entry_id]
    run = archive_data["run"][run_index]

    # 解析 method_ref，例如 /run/0/method/0
    method_index = int(method_ref.split("/")[-1])
    method = run["method"][method_index]

    dft = method.get("dft", {})
    xc = dft.get("xc_functional", {})

    # 將 XC 設定轉成文字，以便比較
    xc_text = json.dumps(
        xc,
        sort_keys=True,
        ensure_ascii=False
    )

    metadata = archive_data.get("metadata", {})

    method_records.append({
        "entry_id": entry_id,
        "energy_eV_atom": row["energy_eV_atom"],
        "energy_group": (
            "Low-energy group"
            if row["energy_eV_atom"] < -5.0
            else "High-energy group"
        ),
        "xc_functional": xc_text,
        "upload_id": metadata.get("upload_id"),
        "n_atoms": row["n_atoms"]
    })

method_df = pd.DataFrame(method_records)

print("=== MASnI3 DFT Energy Group Analysis ===")

print("\n=== Energy Group Distribution ===")
print(method_df["energy_group"].value_counts())

print("\n=== Energy Statistics by Group ===")
print(
    method_df.groupby("energy_group")["energy_eV_atom"]
    .agg(["count", "mean", "std", "min", "max"])
)

print("\n=== XC Functional Distribution ===")
for xc_name, count in method_df["xc_functional"].value_counts().items():
    print("\nCount:", count)
    print("XC:", xc_name[:1000])

print("\n=== Energy Group vs XC Functional ===")
print(
    pd.crosstab(
        method_df["energy_group"],
        method_df["xc_functional"]
    ).to_string()
)

print("\n=== Upload ID Distribution ===")
print(
    pd.crosstab(
        method_df["energy_group"],
        method_df["upload_id"]
    ).to_string()
)

=== MASnI3 DFT Energy Group Analysis ===

=== Energy Group Distribution ===
energy_group
High-energy group    39
Low-energy group     16
Name: count, dtype: int64

=== Energy Statistics by Group ===
                   count      mean       std       min       max
energy_group                                                    
High-energy group     39 -3.709857  0.208159 -4.409096 -3.594969
Low-energy group      16 -5.989690  0.001351 -5.991275 -5.987425

=== XC Functional Distribution ===

Count: 36
XC: {"name": "unavailable"}

Count: 19
XC: {"hybrid": [{"name": "HYB_GGA_XC_HSE06"}], "name": "HYB_GGA_XC_HSE06"}

=== Energy Group vs XC Functional ===
xc_functional      {"hybrid": [{"name": "HYB_GGA_XC_HSE06"}], "name": "HYB_GGA_XC_HSE06"}  {"name": "unavailable"}
energy_group                                                                                                      
High-energy group                                                                       3                      

**如果計算設定存在根本差異，需要將兩組資料分開處理。**

In [ ]:
# =====================================================
# Step 13: Compare DFT Settings Between Uploads
# =====================================================

import json

upload_groups = method_df.groupby("upload_id")

for upload_id, group in upload_groups:

    entry_id = group.iloc[0]["entry_id"]

    archive_data = vasp_archives[entry_id]
    run = archive_data["run"][0]
    method = run["method"][0]

    print("\n" + "=" * 60)
    print("Upload ID:", upload_id)
    print("Representative Entry ID:", entry_id)
    print("Configurations:", len(group))

    print("\n=== Energy Range ===")
    print(
        "Min:",
        round(group["energy_eV_atom"].min(), 6),
        "eV/atom"
    )
    print(
        "Max:",
        round(group["energy_eV_atom"].max(), 6),
        "eV/atom"
    )

    print("\n=== DFT Functional ===")
    print(
        json.dumps(
            method.get("dft", {}),
            indent=2,
            ensure_ascii=False
        )[:1500]
    )

    print("\n=== Electronic Settings ===")
    print(
        json.dumps(
            method.get("electronic", {}),
            indent=2,
            ensure_ascii=False
        )[:1500]
    )

    print("\n=== Basis Set ===")
    print(
        json.dumps(
            method.get("basis_set", {}),
            indent=2,
            ensure_ascii=False
        )[:1500]
    )

    print("\n=== Atomic Parameters ===")
    atom_parameters = method.get("atom_parameters", [])

    print("Number of atom parameter records:", len(atom_parameters))

    for atom in atom_parameters[:5]:
        print(
            json.dumps(
                atom,
                ensure_ascii=False
            )[:500]
        )


Upload ID: PACv39BKSFKLtphwxzY-Og
Representative Entry ID: -bc76c6qQnD_laqNST6-8DVJkY8y
Configurations: 39

=== Energy Range ===
Min: -4.409096 eV/atom
Max: -3.594969 eV/atom

=== DFT Functional ===
{
  "xc_functional": {
    "name": "unavailable"
  }
}

=== Electronic Settings ===
{
  "method": "DFT"
}

=== Basis Set ===
[
  {
    "type": "plane waves",
    "cell_dependent": [
      {
        "kind": "plane waves",
        "planewave_cutoff": 8.331318496799999e-17
      }
    ]
  }
]

=== Atomic Parameters ===
Number of atom parameter records: 5
{"atom_number": 53, "n_valence_electrons": 7.0, "label": "I", "mass": 2.107290497078064e-25, "pseudopotential_name": "PAW_PBE I 08Apr2002"}
{"atom_number": 1, "n_valence_electrons": 1.0, "label": "H", "mass": 1.6605390666e-27, "pseudopotential_name": "PAW_PBE H 15Jun2001"}
{"atom_number": 6, "n_valence_electrons": 4.0, "label": "C", "mass": 1.99447347289326e-26, "pseudopotential_name": "PAW_PBE C 08Apr2002"}
{"atom_number": 50, "n_valence_ele

**檢查 55 組資料的完全相同結構**

In [ ]:
import numpy as np
import pandas as pd
import hashlib

# =====================================================
# Step 14: Check Exact/Near-Identical Structures
# =====================================================

structure_records = []

for _, row in validated_df.iterrows():

    entry_id = row["entry_id"]
    run_index = int(row["run_index"])
    calc_index = int(row["calc_index"])

    archive_data = vasp_archives[entry_id]
    run = archive_data["run"][run_index]
    calc = run["calculation"][calc_index]

    # 找到該 Calculation 對應的 System
    system_ref = calc["system_ref"]
    system_index = int(system_ref.split("/")[-1])

    atoms = run["system"][system_index]["atoms"]

    labels = atoms["labels"]

    # NOMAD Archive 使用 SI 單位
    positions = np.asarray(
        atoms["positions"], dtype=float
    ) / 1e-10

    lattice = np.asarray(
        atoms["lattice_vectors"], dtype=float
    ) / 1e-10

    # 四捨五入至 0.0001 Å
    positions_rounded = np.round(positions, 4)
    lattice_rounded = np.round(lattice, 4)

    # 建立結構指紋
    fingerprint_data = (
        str(list(labels))
        + str(positions_rounded.tolist())
        + str(lattice_rounded.tolist())
    )

    structure_hash = hashlib.sha256(
        fingerprint_data.encode()
    ).hexdigest()

    structure_records.append({
        "entry_id": entry_id,
        "run_index": run_index,
        "calc_index": calc_index,
        "energy_eV_atom": row["energy_eV_atom"],
        "structure_hash": structure_hash
    })

structure_df = pd.DataFrame(structure_records)

print("=== MASnI3 Structure Duplication Analysis ===")

print("Total DFT configurations:", len(structure_df))

print(
    "Unique coordinate fingerprints:",
    structure_df["structure_hash"].nunique()
)

print(
    "Repeated configurations:",
    len(structure_df)
    - structure_df["structure_hash"].nunique()
)

print("\n=== Repeated Structure Groups ===")

duplicate_counts = (
    structure_df["structure_hash"]
    .value_counts()
)

print(
    duplicate_counts[
        duplicate_counts > 1
    ].head(20)
)

print("\n=== Entries per Upload ===")

structure_df = structure_df.merge(
    method_df[
        ["entry_id", "energy_eV_atom", "upload_id"]
    ].drop_duplicates(),
    on=["entry_id", "energy_eV_atom"],
    how="left"
)

print(
    structure_df.groupby("upload_id")[
        "structure_hash"
    ].agg(["count", "nunique"])
)

=== MASnI3 Structure Duplication Analysis ===
Total DFT configurations: 55
Unique coordinate fingerprints: 28
Repeated configurations: 27

=== Repeated Structure Groups ===
structure_hash
29a7df73316209dc794a7e524834607996926805cee445437a631919c9e6139e    6
437de00a72427e798110f4ee9d78bce51ea213c4ca75e2af435896f23bfb0bd8    6
1c8795692c48e8aacdc4c2a0fbf0f60681168d3c134d1ab335772b30f0df6a71    6
4bdc956b7dd0b0620915dd2ccf924cca5a72add892a60eabf5e1a888dd233e22    5
608312e886b97e52edc8fd148af5039efd63c954be4ee21c7dc731d34e334f38    5
02e405088c88367ed3a69150b7ec0deba386dd4ad987139bd9a09292d930891a    5
Name: count, dtype: int64

=== Entries per Upload ===
                        count  nunique
upload_id                             
PACv39BKSFKLtphwxzY-Og     39       12
sGc92ghHQ3W0YPtS3DO8qg     16       16


**檢查重複結構是否具有不同 DFT 能量**

In [ ]:
import pandas as pd
import numpy as np

# =====================================================
# Step 15: Energy Consistency for Duplicate Structures
# =====================================================

energy_consistency = (
    structure_df.groupby("structure_hash")
    .agg(
        n_configurations=("entry_id", "size"),
        n_entries=("entry_id", "nunique"),
        energy_min=("energy_eV_atom", "min"),
        energy_max=("energy_eV_atom", "max"),
        energy_std=("energy_eV_atom", "std")
    )
    .reset_index()
)

energy_consistency["energy_range"] = (
    energy_consistency["energy_max"]
    - energy_consistency["energy_min"]
)

# 只顯示重複結構
duplicate_energy = energy_consistency[
    energy_consistency["n_configurations"] > 1
].copy()

print("=== Duplicate Structure Energy Analysis ===")

print("Total structure groups:", len(energy_consistency))
print("Repeated structure groups:", len(duplicate_energy))

print("\n=== Energy Differences Within Duplicate Groups ===")

print(
    duplicate_energy[
        [
            "n_configurations",
            "n_entries",
            "energy_min",
            "energy_max",
            "energy_range"
        ]
    ].sort_values(
        "energy_range",
        ascending=False
    ).to_string(index=False)
)

print("\n=== Energy Consistency Summary ===")

if not duplicate_energy.empty:

    print(
        "Maximum energy difference:",
        duplicate_energy["energy_range"].max(),
        "eV/atom"
    )

    print(
        "Mean energy difference:",
        duplicate_energy["energy_range"].mean(),
        "eV/atom"
    )

    print(
        "Groups with energy difference > 0.01 eV/atom:",
        int((duplicate_energy["energy_range"] > 0.01).sum())
    )

    print(
        "Groups with energy difference > 0.1 eV/atom:",
        int((duplicate_energy["energy_range"] > 0.1).sum())
    )

=== Duplicate Structure Energy Analysis ===
Total structure groups: 28
Repeated structure groups: 6

=== Energy Differences Within Duplicate Groups ===
 n_configurations  n_entries  energy_min  energy_max  energy_range
                6          6   -4.409096   -3.639720      0.769376
                6          6   -4.407478   -3.638442      0.769036
                6          6   -4.408055   -3.640777      0.767278
                5          5   -3.745302   -3.641197      0.104105
                5          5   -3.643292   -3.594969      0.048323
                5          5   -3.643162   -3.607986      0.035175

=== Energy Consistency Summary ===
Maximum energy difference: 0.769375984166667 eV/atom
Mean energy difference: 0.4155488045833335 eV/atom
Groups with energy difference > 0.01 eV/atom: 6
Groups with energy difference > 0.1 eV/atom: 4


**重複結構能量差異的原因**

In [ ]:
import pandas as pd
import json

# =====================================================
# Step 16: Compare DFT Settings for Duplicate Structures
# =====================================================

# 找出能量差異最大的前三組
top_duplicate_hashes = (
    duplicate_energy.sort_values(
        "energy_range",
        ascending=False
    )
    .head(3)["structure_hash"]
    .tolist()
)

for group_index, structure_hash in enumerate(
    top_duplicate_hashes, 1
):

    group = structure_df[
        structure_df["structure_hash"] == structure_hash
    ].copy()

    group = group.sort_values("energy_eV_atom")

    print("\n" + "=" * 65)
    print(f"Duplicate Structure Group {group_index}")
    print("Number of calculations:", len(group))

    for _, row in group.iterrows():

        entry_id = row["entry_id"]
        run_index = int(row["run_index"])
        calc_index = int(row["calc_index"])

        archive_data = vasp_archives[entry_id]
        run = archive_data["run"][run_index]
        calc = run["calculation"][calc_index]

        method_ref = calc.get("method_ref", "")
        method_index = int(method_ref.split("/")[-1])

        method = run["method"][method_index]

        dft = method.get("dft", {})
        electronic = method.get("electronic", {})
        basis_set = method.get("basis_set", [])

        xc = dft.get("xc_functional", {})

        print("\n----------------------------------------")
        print("Entry ID:", entry_id)

        print(
            "Energy:",
            round(row["energy_eV_atom"], 6),
            "eV/atom"
        )

        print("XC Functional:", xc.get("name"))

        print(
            "Electronic settings:",
            json.dumps(
                electronic,
                ensure_ascii=False
            )[:500]
        )

        # 截斷能：J → eV
        try:
            cutoff_j = (
                basis_set[0]["cell_dependent"][0]
                ["planewave_cutoff"]
            )

            cutoff_ev = cutoff_j / 1.602176634e-19

            print(
                "Plane-wave cutoff:",
                round(cutoff_ev, 2),
                "eV"
            )

        except (KeyError, IndexError, TypeError):
            print("Plane-wave cutoff: unavailable")

        print(
            "Calculation converged:",
            calc.get("calculation_converged")
        )


Duplicate Structure Group 1
Number of calculations: 6

----------------------------------------
Entry ID: U5_-Dv1eY6wn-WNwc3SLZDoRmpJc
Energy: -4.409096 eV/atom
XC Functional: HYB_GGA_XC_HSE06
Electronic settings: {"method": "DFT"}
Plane-wave cutoff: 520.0 eV
Calculation converged: None

----------------------------------------
Entry ID: pv3u5kl5o3_gr8Jchnt4ZpmoWGmm
Energy: -3.758222 eV/atom
XC Functional: unavailable
Electronic settings: {"method": "DFT"}
Plane-wave cutoff: 520.0 eV
Calculation converged: None

----------------------------------------
Entry ID: p9BcT06PUF6omN0iIBNtYhV26ewo
Energy: -3.639724 eV/atom
XC Functional: unavailable
Electronic settings: {"method": "DFT"}
Plane-wave cutoff: 520.0 eV
Calculation converged: None

----------------------------------------
Entry ID: 2hlohPZidhoeyUHL_mkyBj-ZHB-T
Energy: -3.639723 eV/atom
XC Functional: unavailable
Electronic settings: {"method": "DFT"}
Plane-wave cutoff: 520.0 eV
Calculation converged: None

-----------------------

**確認是否存在 SOC 與其他電子結構設定**

In [ ]:
import json

# =====================================================
# Step 17: Inspect Additional VASP Calculation Settings
# =====================================================

selected_ids = [
    "U5_-Dv1eY6wn-WNwc3SLZDoRmpJc",
    "pv3u5kl5o3_gr8Jchnt4ZpmoWGmm",
    "p9BcT06PUF6omN0iIBNtYhV26ewo"
]

for entry_id in selected_ids:

    archive_data = vasp_archives[entry_id]
    run = archive_data["run"][0]

    method = run["method"][0]
    calc = run["calculation"][-1]

    metadata = archive_data.get("metadata", {})

    print("\n" + "=" * 65)
    print("Entry ID:", entry_id)

    print("\n=== Main File ===")
    print(metadata.get("mainfile"))

    print("\n=== Method Keys ===")
    print(list(method.keys()))

    print("\n=== Calculation Keys ===")
    print(list(calc.keys()))

    print("\n=== DFT Settings ===")
    print(
        json.dumps(
            method.get("dft", {}),
            indent=2,
            ensure_ascii=False
        )[:2000]
    )

    print("\n=== SCF Settings ===")
    print(
        json.dumps(
            method.get("scf", {}),
            indent=2,
            ensure_ascii=False
        )[:2000]
    )

    print("\n=== Additional VASP Fields ===")

    vasp_fields = {
        key: value
        for key, value in method.items()
        if "vasp" in key.lower()
        or "spin" in key.lower()
        or "soc" in key.lower()
    }

    print(
        json.dumps(
            vasp_fields,
            indent=2,
            ensure_ascii=False
        )[:2000]
    )


Entry ID: U5_-Dv1eY6wn-WNwc3SLZDoRmpJc

=== Main File ===
1346set_HOIPID/0846/bandgap_hse/vasprun.xml

=== Method Keys ===
['dft', 'electronic', 'scf', 'atom_parameters', 'basis_set']

=== Calculation Keys ===
['system_ref', 'method_ref', 'energy', 'forces', 'stress', 'dos_electronic', 'eigenvalues', 'scf_iteration']

=== DFT Settings ===
{
  "xc_functional": {
    "name": "HYB_GGA_XC_HSE06",
    "hybrid": [
      {
        "name": "HYB_GGA_XC_HSE06"
      }
    ]
  }
}

=== SCF Settings ===
{
  "threshold_energy_change": 1.602176634e-23
}

=== Additional VASP Fields ===
{}

Entry ID: pv3u5kl5o3_gr8Jchnt4ZpmoWGmm

=== Main File ===
1346set_HOIPID/0846/bandgap_gga_P/vasprun.xml

=== Method Keys ===
['dft', 'electronic', 'scf', 'atom_parameters', 'basis_set']

=== Calculation Keys ===
['system_ref', 'method_ref', 'energy', 'forces', 'stress', 'dos_electronic', 'band_structure_electronic', 'scf_iteration']

=== DFT Settings ===
{
  "xc_functional": {
    "name": "unavailable"
  }
}

=== 

**依照原始計算資料夾分類 55 組 DFT 資料**

In [ ]:
import pandas as pd
from pathlib import PurePosixPath

# =====================================================
# Step 18: Classify DFT Data by Original Calculation
# =====================================================

classification_records = []

for _, row in structure_df.iterrows():

    entry_id = row["entry_id"]

    archive_data = vasp_archives[entry_id]

    metadata = archive_data.get("metadata", {})
    mainfile = metadata.get("mainfile", "")

    # 取得包含 vasprun.xml 的上層資料夾名稱
    calculation_folder = PurePosixPath(mainfile).parent.name

    classification_records.append({
        "entry_id": entry_id,
        "run_index": row["run_index"],
        "calc_index": row["calc_index"],
        "structure_hash": row["structure_hash"],
        "energy_eV_atom": row["energy_eV_atom"],
        "upload_id": row["upload_id"],
        "calculation_folder": calculation_folder,
        "mainfile": mainfile
    })

classified_df = pd.DataFrame(classification_records)

print("=== MASnI3 DFT Calculation Classification ===")

print("\n=== Calculation Folder Distribution ===")
print(
    classified_df["calculation_folder"].value_counts()
)

print("\n=== Energy Statistics by Calculation Folder ===")
print(
    classified_df.groupby("calculation_folder")["energy_eV_atom"]
    .agg(["count", "mean", "std", "min", "max"])
    .to_string()
)

print("\n=== Unique Structures by Calculation Folder ===")
print(
    classified_df.groupby("calculation_folder")
    ["structure_hash"]
    .agg(["count", "nunique"])
    .to_string()
)

print("\n=== Calculation Folder vs Upload ===")
print(
    pd.crosstab(
        classified_df["calculation_folder"],
        classified_df["upload_id"]
    ).to_string()
)

classified_df.to_csv(
    "MASnI3_DFT_classified.csv",
    index=False
)

print("\nSaved: MASnI3_DFT_classified.csv")

=== MASnI3 DFT Calculation Classification ===

=== Calculation Folder Distribution ===
calculation_folder
dir_Sn1C1N1H6I3    16
relax              12
bandgap_gga_G       6
bandgap_gga_P       6
bandgap_gga_X       6
dielectric          6
bandgap_hse         3
Name: count, dtype: int64

=== Energy Statistics by Calculation Folder ===
                    count      mean       std       min       max
calculation_folder                                               
bandgap_gga_G           6 -3.641297  0.001655 -3.643292 -3.639414
bandgap_gga_P           6 -3.706471  0.089285 -3.822796 -3.594969
bandgap_gga_X           6 -3.641108  0.001464 -3.642786 -3.639272
bandgap_hse             3 -4.408210  0.000820 -4.409096 -4.407478
dielectric              6 -3.640205  0.000990 -3.641197 -3.638442
dir_Sn1C1N1H6I3        16 -5.989690  0.001351 -5.991275 -5.987425
relax                  12 -3.640444  0.000718 -3.641197 -3.638442

=== Unique Structures by Calculation Folder ===
                    co

**分析不同計算資料夾的結構重疊情況**

In [ ]:
import pandas as pd

# =====================================================
# Step 19: Structure Overlap Between Calculation Types
# =====================================================

folders = sorted(
    classified_df["calculation_folder"].unique()
)

# 每個資料夾對應的結構集合
folder_structures = {
    folder: set(
        classified_df.loc[
            classified_df["calculation_folder"] == folder,
            "structure_hash"
        ]
    )
    for folder in folders
}

# 建立結構重疊矩陣
overlap_matrix = pd.DataFrame(
    index=folders,
    columns=folders,
    dtype=int
)

for folder_a in folders:
    for folder_b in folders:

        overlap_matrix.loc[folder_a, folder_b] = len(
            folder_structures[folder_a]
            & folder_structures[folder_b]
        )

print("=== MASnI3 Structure Overlap Matrix ===")

print(overlap_matrix.astype(int).to_string())

print("\n=== Unique Structures by Folder ===")

for folder in folders:

    current = folder_structures[folder]

    other_structures = set()

    for other_folder in folders:

        if other_folder != folder:
            other_structures.update(
                folder_structures[other_folder]
            )

    exclusive = current - other_structures

    print(
        f"{folder}: "
        f"{len(current)} structures, "
        f"{len(exclusive)} exclusive"
    )

print("\n=== Total Unique Structures ===")

all_unique = set().union(
    *folder_structures.values()
)

print("Total unique fingerprints:", len(all_unique))

=== MASnI3 Structure Overlap Matrix ===
                 bandgap_gga_G  bandgap_gga_P  bandgap_gga_X  bandgap_hse  dielectric  dir_Sn1C1N1H6I3  relax
bandgap_gga_G                6              6              6            3           6                0      6
bandgap_gga_P                6              6              6            3           6                0      6
bandgap_gga_X                6              6              6            3           6                0      6
bandgap_hse                  3              3              3            3           3                0      3
dielectric                   6              6              6            3           6                0      6
dir_Sn1C1N1H6I3              0              0              0            0           0               16      0
relax                        6              6              6            3           6                0     12

=== Unique Structures by Folder ===
bandgap_gga_G: 6 structures, 0 exclusive
ba

**分析 28 種 MASnI₃ 結構的相似性**
1. 從已下載的 VASP Archive 重建晶體。
2. 將 NOMAD 的公尺單位轉換為 Å。
3. 使用 pymatgen 比較結構。
4. 將符合設定容差的結構分組。
5. 統計不同資料來源的結構重疊情況。

# **解決安裝環境問題**

In [1]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f'GPU available: {gpu_name}')
else:
    print('WARNING: No GPU detected!')
    print('Go to Runtime > Change runtime type > T4 GPU, then re-run this cell.')
    print('The tutorial will still work on CPU, but generation will be much slower.')

GPU available: Tesla T4


In [2]:
import os, shutil

REPO_URL = 'https://github.com/RyotaroOKabe/APS_demo_SCIGEN.git'
PROJECT_DIR = '/content/APS_demo_SCIGEN'

os.chdir('/content')

if not os.path.exists(os.path.join(PROJECT_DIR, '.git')):
    if os.path.exists(PROJECT_DIR):
        shutil.rmtree(PROJECT_DIR)
    !git clone {REPO_URL} {PROJECT_DIR}

assert os.path.isdir(os.path.join(PROJECT_DIR, 'scigen')), \
    f'Clone failed — scigen/ not found in {PROJECT_DIR}'

print(f'Repository ready: {PROJECT_DIR}')

Cloning into '/content/APS_demo_SCIGEN'...
remote: Enumerating objects: 604, done.
remote: Counting objects: 100% (201/201), done.
remote: Compressing objects: 100% (156/156), done.
remote: Total 604 (delta 100), reused 138 (delta 42), pack-reused 403 (from 1)
Receiving objects: 100% (604/604), 23.71 MiB | 7.76 MiB/s, done.
Resolving deltas: 100% (280/280), done.
Repository ready: /content/APS_demo_SCIGEN


In [3]:
import torch, os

# Detect PyTorch and CUDA versions for compatible wheel installation
_ver_parts = torch.__version__.split('+')[0].split('.')
TORCH_VER = f'{_ver_parts[0]}.{_ver_parts[1]}.0'
if torch.version.cuda:
    CUDA_VER = 'cu' + torch.version.cuda.replace('.', '')
else:
    CUDA_VER = 'cpu'
PYG_WHEEL_URL = f'https://data.pyg.org/whl/torch-{TORCH_VER}+{CUDA_VER}.html'
os.environ['PYG_WHEEL_URL'] = PYG_WHEEL_URL
print(f'PyTorch {torch.__version__}, CUDA {torch.version.cuda or "N/A (CPU)"}')
print(f'PyG wheel index: {PYG_WHEEL_URL}')

PyTorch 2.14.1+cu130, CUDA 13.0
PyG wheel index: https://data.pyg.org/whl/torch-2.14.0+cu130.html


In [4]:
import sys, torch
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch: 2.14.1+cu130
CUDA: 13.0


In [5]:
print(PYG_WHEEL_URL)

https://data.pyg.org/whl/torch-2.14.0+cu130.html


In [6]:
import os
import torch
from pathlib import Path

PROJECT_DIR = "/content/APS_demo_SCIGEN"

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("PYG_WHEEL_URL:", os.environ.get("PYG_WHEEL_URL"))
print("Requirements exists:",
      Path(PROJECT_DIR, "requirements-colab.txt").is_file())

PyTorch: 2.14.1+cu130
CUDA: 13.0
PYG_WHEEL_URL: https://data.pyg.org/whl/torch-2.14.0+cu130.html
Requirements exists: True


In [7]:
%%time

import sys
import subprocess

PROJECT_DIR = "/content/APS_demo_SCIGEN"

# Install PyG core
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "torch_geometric"],
    check=True
)

# Install tutorial dependencies
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "-r", f"{PROJECT_DIR}/requirements-colab.txt"],
    check=True
)

print("Installation completed successfully.")

Installation completed successfully.
CPU times: user 3.65 ms, sys: 433 µs, total: 4.09 ms
Wall time: 1min 14s


In [8]:
import torch
import torch_geometric
import importlib.util

print("=== Environment Check ===")
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("PyG:", torch_geometric.__version__)
print("GPU available:", torch.cuda.is_available())

print("\n=== PyG Extensions ===")

for pkg in ["torch_scatter", "torch_sparse",
            "torch_cluster", "pyg_lib"]:
    print(pkg, importlib.util.find_spec(pkg) is not None)

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


=== Environment Check ===
PyTorch: 2.14.1+cu130
CUDA: 13.0
PyG: 2.8.1
GPU available: True

=== PyG Extensions ===
torch_scatter False
torch_sparse False
torch_cluster False
pyg_lib False


In [9]:
from pathlib import Path

req = Path("/content/APS_demo_SCIGEN/requirements-colab.txt")

for line in req.read_text().splitlines():
    if any(name in line.lower() for name in
           ["torch", "scatter", "sparse", "cluster", "pyg"]):
        print(line)

# PyTorch + CUDA are provided by the Colab runtime.
# torch-scatter, torch-sparse, torch-cluster must be installed separately
# from the PyG wheel index matching your torch+CUDA version — see setup cell.
torch-geometric>=2.3.0


In [10]:
import requests
from bs4 import BeautifulSoup

url = "https://data.pyg.org/whl/torch-2.11.0+cu130.html"

response = requests.get(url, timeout=20)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")
wheels = [a.get_text() for a in soup.find_all("a")]

for package in ["torch_scatter", "torch_sparse", "torch_cluster"]:
    matches = [
        wheel for wheel in wheels
        if package in wheel
        and "cp313" in wheel
        and "linux_x86_64" in wheel
    ]

    print(f"\n{package}:")
    if matches:
        for wheel in matches:
            print("✓", wheel)
    else:
        print("✗ No compatible wheel found")


torch_scatter:
✓ torch_scatter-2.1.2+pt211cu130-cp313-cp313-linux_x86_64.whl

torch_sparse:
✓ torch_sparse-0.6.18+pt211cu130-cp313-cp313-linux_x86_64.whl

torch_cluster:
✓ torch_cluster-1.6.3+pt211cu130-cp313-cp313-linux_x86_64.whl


In [11]:
%pip install --no-index \
    torch-scatter torch-sparse torch-cluster \
    -f https://data.pyg.org/whl/torch-2.11.0+cu130.html

Looking in links: https://data.pyg.org/whl/torch-2.11.0+cu130.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 88.2 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 56.8 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 142.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [torch-cluster]


解決torch_scatter 的 CUDA 擴充模組與目前 Colab 載入的 PyTorch 二進位版本不相容。

In [12]:
import torch
import sys
import subprocess

print("=== Runtime Environment ===")
print("Loaded PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("Python:", sys.version)

print("\n=== Installed PyTorch ===")

subprocess.run([
    sys.executable, "-m", "pip",
    "show", "torch", "torch-scatter"
])

=== Runtime Environment ===
Loaded PyTorch: 2.14.1+cu130
CUDA: 13.0
Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]

=== Installed PyTorch ===


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'show', 'torch', 'torch-scatter'], returncode=0)

確認目前環境版本

In [13]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)

PyTorch: 2.14.1+cu130
CUDA: 13.0


In [14]:
%pip install --index-url https://download.pytorch.org/whl/cu130 \
    "torch==2.11.0+cu130" \
    "torchvision==0.26.0+cu130"

Looking in indexes: https://download.pytorch.org/whl/cu130
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 46.9 MB/s  0:00:06
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 154.4 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 150.0 MB/s  0:00:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 MB 212.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.1/531.1 MB 15.4 MB/s  0:00:18
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 128.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 60.5 MB/s  0:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 61.0 MB/s  0:00:03
  Attempting uninstall: nvidia-cusparselt-cu13
    Found existing installation: nvidia-cusparselt-cu13 0.8.1
    Uninstalling nvidia-cusparselt-cu13-0.8.1:
      Successfully uninstalled nvidia-cusparselt-cu13-0.8.1
  Attempting uninstall: cuda-toolkit
    Found existing installation: cuda-toolkit 13.0.3.0
   

## **執行階段 → 重新啟動工作階段**

確認目前環境版本

In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)

PyTorch: 2.11.0+cu130
CUDA: 13.0


In [2]:
import torch_scatter
import torch_sparse
import torch_cluster

print("torch_scatter:", torch_scatter.__version__)
print("torch_sparse:", torch_sparse.__version__)
print("torch_cluster:", torch_cluster.__version__)

print("Environment verification successful!")

torch_scatter: 2.1.2+pt211cu130
torch_sparse: 0.6.18+pt211cu130
torch_cluster: 1.6.3+pt211cu130
Environment verification successful!


In [3]:
import os, sys

PROJECT_DIR = '/content/APS_demo_SCIGEN'

os.environ['PROJECT_ROOT'] = PROJECT_DIR
os.environ['HYDRA_JOBS'] = PROJECT_DIR
os.environ['WANDB_DIR'] = os.path.join(PROJECT_DIR, 'wandb')
os.makedirs(os.environ['WANDB_DIR'], exist_ok=True)

sys.path.insert(0, PROJECT_DIR)
sys.path.insert(0, os.path.join(PROJECT_DIR, 'script'))

os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')
print('Environment configured.')

Working directory: /content/APS_demo_SCIGEN
Environment configured.


In [4]:
from pathlib import Path

# SCIGEN 專案目錄
PROJECT_DIR = Path("/content/APS_demo_SCIGEN")

# 模型儲存位置
MODEL_DIR = PROJECT_DIR / "models"
MODEL_PATH = MODEL_DIR / "mp_20"

# 確認專案是否存在
print("Project exists:", PROJECT_DIR.is_dir())
print("Model directory:", MODEL_PATH)

Project exists: True
Model directory: /content/APS_demo_SCIGEN/models/mp_20


NOMAND

In [5]:
# =====================================================
# Step 20.1: Check Existing NOMAD Variables
# =====================================================

required_variables = [
    "validated_df",
    "structure_df",
    "classified_df",
    "vasp_archives"
]

print("=== NOMAD Data Availability Check ===")

for name in required_variables:
    if name in globals():
        obj = globals()[name]

        if hasattr(obj, "__len__"):
            print(f"✓ {name}: Available ({len(obj)} records)")
        else:
            print(f"✓ {name}: Available")
    else:
        print(f"✗ {name}: Not found")

=== NOMAD Data Availability Check ===
✗ validated_df: Not found
✗ structure_df: Not found
✗ classified_df: Not found
✗ vasp_archives: Not found


In [6]:
import numpy as np
from pymatgen.core import Structure
from pymatgen.analysis.structure_matcher import StructureMatcher

# =====================================================
# Step 20: MASnI3 Crystal Structure Similarity
# =====================================================

unique_rows = structure_df.drop_duplicates(
    subset="structure_hash"
).reset_index(drop=True)

structures = []
structure_info = []

for _, row in unique_rows.iterrows():

    entry_id = row["entry_id"]
    run_index = int(row["run_index"])
    calc_index = int(row["calc_index"])

    archive_data = vasp_archives[entry_id]
    run = archive_data["run"][run_index]
    calc = run["calculation"][calc_index]

    system_index = int(
        calc["system_ref"].split("/")[-1]
    )

    atoms = run["system"][system_index]["atoms"]

    # SI → Å
    lattice = np.array(
        atoms["lattice_vectors"],
        dtype=float
    ) / 1e-10

    positions = np.array(
        atoms["positions"],
        dtype=float
    ) / 1e-10

    structure = Structure(
        lattice=lattice,
        species=atoms["labels"],
        coords=positions,
        coords_are_cartesian=True
    )

    structures.append(structure)

    structure_info.append({
        "entry_id": entry_id,
        "structure_hash": row["structure_hash"],
        "upload_id": row["upload_id"]
    })


# 使用較嚴格的結構比對容差
matcher = StructureMatcher(
    ltol=0.02,
    stol=0.05,
    angle_tol=1,
    primitive_cell=False,
    scale=False
)

# 建立相似結構分組
groups = []

for i, structure in enumerate(structures):

    matched = False

    for group in groups:

        representative = structures[group[0]]

        if matcher.fit(representative, structure):
            group.append(i)
            matched = True
            break

    if not matched:
        groups.append([i])


print("=== MASnI3 StructureMatcher Analysis ===")

print("Original coordinate fingerprints:", len(structures))
print("StructureMatcher groups:", len(groups))

print("\n=== Group Sizes ===")

for i, group in enumerate(groups, 1):

    uploads = {
        structure_info[index]["upload_id"]
        for index in group
    }

    print(
        f"Group {i}: "
        f"{len(group)} structures, "
        f"{len(uploads)} upload source(s)"
    )

print("\n=== Summary ===")

print("Unique coordinate fingerprints:", len(structures))
print("Similarity groups:", len(groups))

print(
    "Structures grouped as similar:",
    len(structures) - len(groups)
)

NameError: name 'structure_df' is not defined

重新執行階段>>不用重新載入622筆資料>>csv檔有存

In [ ]:
# =====================================================
# Step 1: Check Saved MASnI3 DFT Files
# =====================================================

from pathlib import Path

search_dirs = [
    Path("/content"),
    Path("/content/APS_demo_SCIGEN"),
    Path("/content/drive/MyDrive")
]

target_files = [
    "MASnI3_DFT_availability_updated.csv",
    "MASnI3_VASP_validated_EFS.csv",
    "MASnI3_DFT_classified.csv",
    "MASnI3_failed_entries.csv"
]

print("=== MASnI3 DFT File Recovery Check ===")

for filename in target_files:
    found = []

    for directory in search_dirs:
        if directory.exists():
            found.extend(directory.rglob(filename))

    if found:
        print(f"\n✓ {filename}")
        for path in found[:5]:
            print("  Location:", path)
    else:
        print(f"\n✗ {filename}: Not found")

=== MASnI3 DFT File Recovery Check ===

✓ MASnI3_DFT_availability_updated.csv
  Location: /content/MASnI3_DFT_availability_updated.csv

✓ MASnI3_VASP_validated_EFS.csv
  Location: /content/MASnI3_VASP_validated_EFS.csv

✓ MASnI3_DFT_classified.csv
  Location: /content/MASnI3_DFT_classified.csv

✓ MASnI3_failed_entries.csv
  Location: /content/MASnI3_failed_entries.csv


**重新載入已儲存的 DFT 資料**

In [ ]:
# =====================================================
# Step 1: Restore MASnI3 DFT DataFrames
# =====================================================

import pandas as pd

availability_df = pd.read_csv(
    "/content/MASnI3_DFT_availability_updated.csv"
)

validated_df = pd.read_csv(
    "/content/MASnI3_VASP_validated_EFS.csv"
)

classified_df = pd.read_csv(
    "/content/MASnI3_DFT_classified.csv"
)

print("=== MASnI3 DFT Data Recovery ===")

print("NOMAD entries:", len(availability_df))
print("Validated configurations:", len(validated_df))
print("Classified configurations:", len(classified_df))

print("\n=== Validated DataFrame Columns ===")
print(validated_df.columns.tolist())

print("\n=== Classified DataFrame Columns ===")
print(classified_df.columns.tolist())

=== MASnI3 DFT Data Recovery ===
NOMAD entries: 622
Validated configurations: 55
Classified configurations: 55

=== Validated DataFrame Columns ===
['entry_id', 'run_index', 'calc_index', 'system_ref', 'method_ref', 'n_atoms', 'energy_eV', 'energy_eV_atom', 'max_force_eV_A', 'max_abs_stress_GPa', 'valid_EFS']

=== Classified DataFrame Columns ===
['entry_id', 'run_index', 'calc_index', 'structure_hash', 'energy_eV_atom', 'upload_id', 'calculation_folder', 'mainfile']


**恢復 structure_df**

In [ ]:
# =====================================================
# Step 2: Restore MASnI3 Structure DataFrame
# =====================================================

structure_df = classified_df[
    [
        "entry_id",
        "run_index",
        "calc_index",
        "structure_hash",
        "upload_id"
    ]
].copy()

print("=== MASnI3 Structure Data Recovery ===")

print("Total DFT configurations:", len(structure_df))

print(
    "Unique coordinate fingerprints:",
    structure_df["structure_hash"].nunique()
)

print(
    "Repeated configurations:",
    len(structure_df) - structure_df["structure_hash"].nunique()
)

print("\n=== Upload Distribution ===")

print(
    structure_df.groupby("upload_id")["structure_hash"]
    .agg(["count", "nunique"])
)

=== MASnI3 Structure Data Recovery ===
Total DFT configurations: 55
Unique coordinate fingerprints: 28
Repeated configurations: 27

=== Upload Distribution ===
                        count  nunique
upload_id                             
PACv39BKSFKLtphwxzY-Og     39       12
sGc92ghHQ3W0YPtS3DO8qg     16       16


**下載 28 個代表結構的 NOMAD Archive**

In [ ]:
# =====================================================
# Step 3: Recover 28 Representative VASP Archives
# =====================================================

import requests
import time

API_URL = "https://nomad-lab.eu/prod/v1/api/v1"

# 每種座標指紋保留一筆代表資料
representatives = structure_df.drop_duplicates(
    subset="structure_hash"
).reset_index(drop=True)

vasp_archives = {}
failed_ids = []

session = requests.Session()

for i, row in representatives.iterrows():

    entry_id = row["entry_id"]

    url = f"{API_URL}/entries/{entry_id}/archive"

    for attempt in range(5):
        try:
            response = session.get(url, timeout=60)

            if response.status_code == 429:
                retry_after = response.headers.get("Retry-After")
                wait = (
                    float(retry_after)
                    if retry_after is not None
                    else min(10 * (2 ** attempt), 120)
                )
                print(f"Rate limited. Waiting {wait:.0f} seconds...")
                time.sleep(wait)
                continue

            response.raise_for_status()

            vasp_archives[entry_id] = response.json()["data"]["archive"]
            break

        except (requests.RequestException, KeyError, ValueError) as e:
            print(f"Attempt {attempt + 1} failed: {entry_id} — {e}")
            time.sleep(3)

    else:
        failed_ids.append(entry_id)

    print(
        f"Processed: {i + 1}/{len(representatives)} | "
        f"Successful: {len(vasp_archives)}"
    )

    time.sleep(2)

print("\n=== Archive Recovery Summary ===")
print("Representative structures:", len(representatives))
print("Successfully recovered:", len(vasp_archives))
print("Failed:", len(failed_ids))

if failed_ids:
    print("Failed Entry IDs:", failed_ids)

Processed: 1/28 | Successful: 1
Processed: 2/28 | Successful: 2
Processed: 3/28 | Successful: 3
Processed: 4/28 | Successful: 4
Processed: 5/28 | Successful: 5
Processed: 6/28 | Successful: 6
Processed: 7/28 | Successful: 7
Processed: 8/28 | Successful: 7
Processed: 9/28 | Successful: 8
Processed: 10/28 | Successful: 8
Processed: 11/28 | Successful: 8
Processed: 12/28 | Successful: 8
Processed: 13/28 | Successful: 9
Processed: 14/28 | Successful: 9
Processed: 15/28 | Successful: 9
Processed: 16/28 | Successful: 9
Processed: 17/28 | Successful: 9
Processed: 18/28 | Successful: 9
Processed: 19/28 | Successful: 9
Processed: 20/28 | Successful: 9
Processed: 21/28 | Successful: 9
Processed: 22/28 | Successful: 9
Processed: 23/28 | Successful: 9
Processed: 24/28 | Successful: 9
Processed: 25/28 | Successful: 9
Processed: 26/28 | Successful: 9
Processed: 27/28 | Successful: 9
Processed: 28/28 | Successful: 9

=== Archive Recovery Summary ===
Representative structures: 28
Successfully recovere

In [ ]:
# =====================================================
# Step 3.1: Verify Representative Archive Coverage
# =====================================================

representatives = structure_df.drop_duplicates(
    subset="structure_hash"
).reset_index(drop=True)

required_ids = set(representatives["entry_id"])
downloaded_ids = set(vasp_archives.keys())

missing_ids = required_ids - downloaded_ids

print("=== NOMAD Archive Coverage Verification ===")

print("Representative structures:", len(representatives))
print("Unique Entry IDs required:", len(required_ids))
print("Archives stored:", len(downloaded_ids))

print("\n=== Coverage ===")
print("Required Entry IDs recovered:", len(required_ids & downloaded_ids))
print("Missing Entry IDs:", len(missing_ids))

if missing_ids:
    print("\nMissing IDs:")
    for entry_id in sorted(missing_ids):
        print(entry_id)
else:
    print("\nAll required Entry IDs are available!")

print("\n=== Configurations per Entry ID ===")

print(
    representatives.groupby("entry_id")
    .size()
    .value_counts()
    .sort_index()
)

=== NOMAD Archive Coverage Verification ===
Representative structures: 28
Unique Entry IDs required: 9
Archives stored: 9

=== Coverage ===
Required Entry IDs recovered: 9
Missing Entry IDs: 0

All required Entry IDs are available!

=== Configurations per Entry ID ===
1     6
2     1
4     1
16    1
Name: count, dtype: int64


NOMAD 中原本的 28 個 MASnI₃ 座標指紋，在目前的 StructureMatcher 條件下，可歸為 8 組相似晶體結構。

In [ ]:
# =====================================================
# Step 20: MASnI3 Crystal Structure Similarity
# =====================================================

import numpy as np
from pymatgen.core import Structure
from pymatgen.analysis.structure_matcher import StructureMatcher

# 每種座標指紋取一筆代表結構
unique_rows = structure_df.drop_duplicates(
    subset="structure_hash"
).reset_index(drop=True)

structures = []
structure_info = []

for _, row in unique_rows.iterrows():

    entry_id = row["entry_id"]
    run_index = int(row["run_index"])
    calc_index = int(row["calc_index"])

    archive_data = vasp_archives[entry_id]
    run = archive_data["run"][run_index]
    calc = run["calculation"][calc_index]

    # 找到與 DFT calculation 對應的結構
    system_index = int(
        calc["system_ref"].split("/")[-1]
    )

    atoms = run["system"][system_index]["atoms"]

    # NOMAD SI 單位轉換為 Å
    lattice = np.array(
        atoms["lattice_vectors"],
        dtype=float
    ) / 1e-10

    positions = np.array(
        atoms["positions"],
        dtype=float
    ) / 1e-10

    structure = Structure(
        lattice=lattice,
        species=atoms["labels"],
        coords=positions,
        coords_are_cartesian=True
    )

    structures.append(structure)

    structure_info.append({
        "entry_id": entry_id,
        "structure_hash": row["structure_hash"],
        "upload_id": row["upload_id"]
    })

# 使用嚴格的晶體結構比對條件
matcher = StructureMatcher(
    ltol=0.02,
    stol=0.05,
    angle_tol=1,
    primitive_cell=False,
    scale=False
)

groups = []

for i, structure in enumerate(structures):

    matched = False

    for group in groups:

        representative = structures[group[0]]

        if matcher.fit(representative, structure):
            group.append(i)
            matched = True
            break

    if not matched:
        groups.append([i])

print("=== MASnI3 StructureMatcher Analysis ===")

print("Original coordinate fingerprints:", len(structures))
print("StructureMatcher groups:", len(groups))

print("\n=== Group Sizes ===")

for i, group in enumerate(groups, 1):

    uploads = {
        structure_info[index]["upload_id"]
        for index in group
    }

    print(
        f"Group {i}: "
        f"{len(group)} structures, "
        f"{len(uploads)} upload source(s)"
    )

print("\n=== Summary ===")

print("Unique coordinate fingerprints:", len(structures))
print("Similarity groups:", len(groups))

print(
    "Structures grouped as similar:",
    len(structures) - len(groups)
)

=== MASnI3 StructureMatcher Analysis ===
Original coordinate fingerprints: 28
StructureMatcher groups: 8

=== Group Sizes ===
Group 1: 1 structures, 1 upload source(s)
Group 2: 1 structures, 1 upload source(s)
Group 3: 3 structures, 1 upload source(s)
Group 4: 1 structures, 1 upload source(s)
Group 5: 1 structures, 1 upload source(s)
Group 6: 5 structures, 1 upload source(s)
Group 7: 13 structures, 1 upload source(s)
Group 8: 3 structures, 1 upload source(s)

=== Summary ===
Unique coordinate fingerprints: 28
Similarity groups: 8
Structures grouped as similar: 20


確認 8 組結構分別來自哪個資料來源

In [ ]:
# =====================================================
# Step 21: MASnI3 Similarity Group Source Analysis
# =====================================================

import pandas as pd

group_records = []

for group_id, group in enumerate(groups, start=1):

    for index in group:

        info = structure_info[index]

        group_records.append({
            "group_id": group_id,
            "entry_id": info["entry_id"],
            "structure_hash": info["structure_hash"],
            "upload_id": info["upload_id"]
        })

group_df = pd.DataFrame(group_records)

print("=== MASnI3 Structure Group Distribution ===")

print(
    group_df.groupby(["group_id", "upload_id"])
    .size()
    .unstack(fill_value=0)
)

print("\n=== Summary ===")
print("Similarity groups:", group_df["group_id"].nunique())
print("Total fingerprints:", len(group_df))
print("Upload sources:", group_df["upload_id"].nunique())

=== MASnI3 Structure Group Distribution ===
upload_id  PACv39BKSFKLtphwxzY-Og  sGc92ghHQ3W0YPtS3DO8qg
group_id                                                 
1                               1                       0
2                               1                       0
3                               3                       0
4                               1                       0
5                               1                       0
6                               5                       0
7                               0                      13
8                               0                       3

=== Summary ===
Similarity groups: 8
Total fingerprints: 28
Upload sources: 2


檢查 Relaxation DFT 資料品質

In [ ]:
# =====================================================
# Step 22: MASnI3 Relaxation DFT Quality Check
# =====================================================

relax_df = classified_df[
    classified_df["calculation_folder"] == "relax"
].copy()

relax_df = relax_df.merge(
    validated_df[
        [
            "entry_id",
            "run_index",
            "calc_index",
            "energy_eV_atom",
            "max_force_eV_A",
            "max_abs_stress_GPa",
            "valid_EFS"
        ]
    ],
    on=["entry_id", "run_index", "calc_index"],
    how="left",
    suffixes=("_classified", "_validated"),
    validate="one_to_one"
)

print("=== MASnI3 Relaxation DFT Quality ===")

print("Relax configurations:", len(relax_df))
print("Unique coordinate fingerprints:", relax_df["structure_hash"].nunique())

print("\n=== EFS Availability ===")
print(relax_df["valid_EFS"].value_counts(dropna=False))

print("\n=== Maximum Force Distribution (eV/Å) ===")
print(relax_df["max_force_eV_A"].describe())

print("\n=== Maximum Absolute Stress Distribution (GPa) ===")
print(relax_df["max_abs_stress_GPa"].describe())

print("\n=== Relaxation Configurations ===")
print(
    relax_df[
        [
            "entry_id",
            "energy_eV_atom_validated",
            "max_force_eV_A",
            "max_abs_stress_GPa",
            "valid_EFS"
        ]
    ].to_string(index=False)
)

=== MASnI3 Relaxation DFT Quality ===
Relax configurations: 12
Unique coordinate fingerprints: 12

=== EFS Availability ===
valid_EFS
True    12
Name: count, dtype: int64

=== Maximum Force Distribution (eV/Å) ===
count    12.000000
mean      0.011324
std       0.002991
min       0.007971
25%       0.009003
50%       0.010288
75%       0.013167
max       0.016697
Name: max_force_eV_A, dtype: float64

=== Maximum Absolute Stress Distribution (GPa) ===
count    12.000000
mean      0.014012
std       0.008923
min       0.003532
25%       0.009403
50%       0.011017
75%       0.018757
max       0.035272
Name: max_abs_stress_GPa, dtype: float64

=== Relaxation Configurations ===
                    entry_id  energy_eV_atom_validated  max_force_eV_A  max_abs_stress_GPa  valid_EFS
0hlSL9bz6yPH2CmO-NRbrr_vK8U6                 -3.640576        0.008390            0.010979       True
4WF5u7xlskuyKBUyEn9AM2VT81fV                 -3.639720        0.007971            0.010713       True
ETs3KhaeW_G

檢查 Relaxation DFT 計算條件

In [ ]:
# =====================================================
# Step 23.1: Recover Missing Relaxation Archives
# =====================================================

import requests
import time

API_URL = "https://nomad-lab.eu/prod/v1/api/v1"

required_ids = set(relax_df["entry_id"])
existing_ids = set(vasp_archives.keys())

missing_ids = sorted(required_ids - existing_ids)

print("=== MASnI3 Relaxation Archive Recovery ===")
print("Required Entry IDs:", len(required_ids))
print("Already available:", len(required_ids & existing_ids))
print("Missing Entry IDs:", len(missing_ids))

session = requests.Session()
failed_ids = []

for i, entry_id in enumerate(missing_ids, start=1):

    url = f"{API_URL}/entries/{entry_id}/archive"
    success = False

    for attempt in range(5):

        try:
            response = session.get(url, timeout=60)

            if response.status_code == 429:
                retry_after = response.headers.get("Retry-After")
                wait = (
                    float(retry_after)
                    if retry_after is not None
                    else min(10 * 2**attempt, 120)
                )
                print(f"Rate limit (429): waiting {wait:.0f}s")
                time.sleep(wait)
                continue

            response.raise_for_status()

            vasp_archives[entry_id] = response.json()["data"]["archive"]
            success = True
            break

        except (requests.RequestException, KeyError, ValueError) as e:
            print(f"Attempt {attempt + 1}: {e}")
            time.sleep(3)

    if not success:
        failed_ids.append(entry_id)

    print(
        f"Processed: {i}/{len(missing_ids)} | "
        f"Success: {success}"
    )

    time.sleep(2)

print("\n=== Final Verification ===")

remaining = required_ids - set(vasp_archives.keys())

print("Required Entry IDs:", len(required_ids))
print("Recovered Entry IDs:", len(required_ids - remaining))
print("Remaining missing:", len(remaining))

if remaining:
    print("Missing IDs:", sorted(remaining))
else:
    print("All relaxation archives recovered successfully!")

=== MASnI3 Relaxation Archive Recovery ===
Required Entry IDs: 6
Already available: 3
Missing Entry IDs: 3
Processed: 1/3 | Success: True
Processed: 2/3 | Success: True
Processed: 3/3 | Success: True

=== Final Verification ===
Required Entry IDs: 6
Recovered Entry IDs: 6
Remaining missing: 0
All relaxation archives recovered successfully!


In [ ]:
# =====================================================
# Step 23: MASnI3 Relaxation DFT Method Verification
# =====================================================

import pandas as pd
import json

records = []

for _, row in relax_df.iterrows():

    entry_id = row["entry_id"]
    run_index = int(row["run_index"])
    calc_index = int(row["calc_index"])

    archive = vasp_archives[entry_id]
    run = archive["run"][run_index]
    calc = run["calculation"][calc_index]

    method_index = int(
        calc["method_ref"].split("/")[-1]
    )

    method = run["method"][method_index]

    xc = method.get("dft", {}).get(
        "xc_functional", {}
    )

    scf = method.get("scf", {})

    records.append({
        "entry_id": entry_id,
        "calc_index": calc_index,
        "xc_functional": json.dumps(xc, sort_keys=True),
        "scf_threshold_J": scf.get(
            "threshold_energy_change"
        ),
        "calculation_converged": calc.get(
            "calculation_converged"
        ),
        "n_scf_iterations": calc.get(
            "n_scf_iterations"
        ),
        "mainfile": archive.get(
            "metadata", {}
        ).get("mainfile")
    })

method_check_df = pd.DataFrame(records)

print("=== MASnI3 Relaxation Method Verification ===")

print("\n=== XC Functional Distribution ===")
print(method_check_df["xc_functional"].value_counts())

print("\n=== SCF Threshold Distribution ===")
print(method_check_df["scf_threshold_J"].value_counts(dropna=False))

print("\n=== Convergence Status ===")
print(method_check_df["calculation_converged"].value_counts(dropna=False))

print("\n=== Detailed Results ===")
print(method_check_df.to_string(index=False))

=== MASnI3 Relaxation Method Verification ===

=== XC Functional Distribution ===
xc_functional
{"name": "unavailable"}    12
Name: count, dtype: int64

=== SCF Threshold Distribution ===
scf_threshold_J
1.602177e-23    12
Name: count, dtype: int64

=== Convergence Status ===
calculation_converged
None    12
Name: count, dtype: int64

=== Detailed Results ===
                    entry_id  calc_index           xc_functional  scf_threshold_J calculation_converged n_scf_iterations                              mainfile
0hlSL9bz6yPH2CmO-NRbrr_vK8U6           0 {"name": "unavailable"}     1.602177e-23                  None             None 1346set_HOIPID/0843/relax/vasprun.xml
4WF5u7xlskuyKBUyEn9AM2VT81fV           0 {"name": "unavailable"}     1.602177e-23                  None             None 1346set_HOIPID/0846/relax/vasprun.xml
ETs3KhaeW_GioQu8DvV_r_TdXXe4           0 {"name": "unavailable"}     1.602177e-23                  None             None 1346set_HOIPID/0844/relax/vasprun.xml
ET

In [ ]:
# =====================================================
# Step 24: Inspect VASP Original Calculation Settings
# =====================================================

import json

entry_id = relax_df.iloc[0]["entry_id"]
archive = vasp_archives[entry_id]

run = archive["run"][0]
method = run["method"][0]

print("=== MASnI3 VASP Method Inspection ===")

print("Entry ID:", entry_id)

print("\n=== Method Keys ===")
print(list(method.keys()))

print("\n=== DFT Settings ===")
print(json.dumps(
    method.get("dft", {}),
    indent=2,
    default=str
))

print("\n=== Electronic Settings ===")
print(json.dumps(
    method.get("electronic", {}),
    indent=2,
    default=str
))

print("\n=== SCF Settings ===")
print(json.dumps(
    method.get("scf", {}),
    indent=2,
    default=str
))

print("\n=== Run Keys ===")
print(list(run.keys()))

print("\n=== Metadata Mainfile ===")
print(archive.get("metadata", {}).get("mainfile"))

=== MASnI3 VASP Method Inspection ===
Entry ID: 0hlSL9bz6yPH2CmO-NRbrr_vK8U6

=== Method Keys ===
['dft', 'electronic', 'scf', 'atom_parameters', 'basis_set']

=== DFT Settings ===
{
  "xc_functional": {
    "name": "unavailable"
  }
}

=== Electronic Settings ===
{
  "method": "DFT"
}

=== SCF Settings ===
{
  "threshold_energy_change": 1.602176634e-23
}

=== Run Keys ===
['program', 'method', 'system', 'calculation']

=== Metadata Mainfile ===
1346set_HOIPID/0843/relax/vasprun.xml


讀取原始 VASP 計算參數

In [ ]:
# =====================================================
# Step 25: Inspect Original MASnI3 VASP XML
# =====================================================

import requests
import xml.etree.ElementTree as ET

entry_id = "0hlSL9bz6yPH2CmO-NRbrr_vK8U6"

row = classified_df[
    classified_df["entry_id"] == entry_id
].iloc[0]

upload_id = row["upload_id"]
mainfile = row["mainfile"]

API_URL = "https://nomad-lab.eu/prod/v1/api/v1"

url = f"{API_URL}/uploads/{upload_id}/raw/{mainfile}"

print("=== Original VASP File ===")
print("Entry ID:", entry_id)
print("Upload ID:", upload_id)
print("Mainfile:", mainfile)

response = requests.get(url, timeout=120)

print("\nHTTP Status:", response.status_code)
print("Downloaded bytes:", len(response.content))

response.raise_for_status()

root = ET.fromstring(response.content)

# 讀取 VASP INCAR 和 parameters
incar = root.find("incar")
parameters = root.find("parameters")

def read_vasp_parameters(section):
    values = {}

    if section is None:
        return values

    for element in section.iter():
        name = element.attrib.get("name")

        if name and element.text:
            values[name] = element.text.strip()

    return values

incar_params = read_vasp_parameters(incar)
all_params = read_vasp_parameters(parameters)

important_keys = [
    "GGA",
    "METAGGA",
    "LHFCALC",
    "HFSCREEN",
    "AEXX",
    "ENCUT",
    "EDIFF",
    "EDIFFG",
    "ISPIN",
    "LSORBIT",
    "ISMEAR",
    "SIGMA",
    "IBRION",
    "NSW",
    "ISIF",
    "NELM"
]

print("\n=== Original INCAR Parameters ===")

for key in important_keys:
    value = incar_params.get(
        key,
        all_params.get(key, "Not found")
    )

    print(f"{key}: {value}")

print("\n=== Ionic Calculation Steps ===")

ionic_steps = root.findall("calculation")

print("Number of ionic steps:", len(ionic_steps))

=== Original VASP File ===
Entry ID: 0hlSL9bz6yPH2CmO-NRbrr_vK8U6
Upload ID: PACv39BKSFKLtphwxzY-Og
Mainfile: 1346set_HOIPID/0843/relax/vasprun.xml

HTTP Status: 200
Downloaded bytes: 71368

=== Original INCAR Parameters ===
GGA: ML
METAGGA: Not found
LHFCALC: F
HFSCREEN: 0.00000000
AEXX: 0.00000000
ENCUT: 400.00000000
EDIFF: 0.00010000
EDIFFG: -0.01000000
ISPIN: 1
LSORBIT: F
ISMEAR: 0
SIGMA: 0.01000000
IBRION: 1
NSW: 100
ISIF: 3
NELM: 1

=== Ionic Calculation Steps ===
Number of ionic steps: 1


In [ ]:
# =====================================================
# Step 26: Verify VASP Electronic Calculation Mode
# =====================================================

important_settings = [
    "GGA",
    "LUSE_VDW",
    "IVDW",
    "IVDW_NL",
    "AGGAC",
    "ZAB_VDW",
    "ALGO",
    "IALGO",
    "ISTART",
    "ICHARG",
    "NELM",
    "NELMIN",
    "NELMDL",
    "LDIAG",
    "EDIFF",
    "EDIFFG",
    "NSW",
    "IBRION",
    "ISIF"
]

print("=== MASnI3 Electronic Calculation Settings ===")

for key in important_settings:
    value = incar_params.get(
        key,
        all_params.get(key, "Not found")
    )
    print(f"{key}: {value}")

print("\n=== Electronic Iterations per Ionic Step ===")

calculations = root.findall("calculation")

for i, calculation in enumerate(calculations):
    scsteps = calculation.findall("scstep")

    print(
        f"Ionic step {i + 1}: "
        f"{len(scsteps)} electronic iterations"
    )

print("\n=== Available Calculation Sections ===")

if calculations:
    print([
        child.tag
        for child in calculations[-1]
    ])

=== MASnI3 Electronic Calculation Settings ===
GGA: ML
LUSE_VDW: T
IVDW: Not found
IVDW_NL: Not found
AGGAC: 0.00000000
ZAB_VDW: Not found
ALGO: Not found
IALGO: 38
ISTART: 0
ICHARG: 2
NELM: 1
NELMIN: 2
NELMDL: -5
LDIAG: T
EDIFF: 0.00010000
EDIFFG: -0.01000000
NSW: 100
IBRION: 1
ISIF: 3

=== Electronic Iterations per Ionic Step ===
Ionic step 1: 11 electronic iterations

=== Available Calculation Sections ===
['scstep', 'scstep', 'scstep', 'scstep', 'scstep', 'scstep', 'scstep', 'scstep', 'scstep', 'scstep', 'scstep', 'structure', 'varray', 'varray', 'energy', 'time', 'eigenvalues', 'separator']


確認 NELM 與電子收斂資訊

In [ ]:
# =====================================================
# Step 27: Verify Actual VASP Electronic Iterations
# =====================================================

print("=== NELM Parameter Verification ===")

for section_name in ["incar", "parameters"]:

    section = root.find(section_name)

    print(f"\n=== {section_name.upper()} ===")

    if section is None:
        print("Section not found")
        continue

    for element in section.iter():

        if element.attrib.get("name") in [
            "NELM", "NELMIN", "NELMDL",
            "EDIFF", "IALGO", "ALGO"
        ]:
            print(
                element.attrib.get("name"),
                "=",
                element.text.strip() if element.text else None
            )

print("\n=== Electronic Step Energy Changes ===")

calculations = root.findall("calculation")

if calculations:

    scsteps = calculations[-1].findall("scstep")

    for i, step in enumerate(scsteps, 1):

        energy_section = step.find("energy")

        values = {}

        if energy_section is not None:
            for item in energy_section.findall("i"):

                name = item.attrib.get("name")

                if item.text:
                    values[name] = item.text.strip()

        print(
            f"SCF Step {i}:",
            values
        )

=== NELM Parameter Verification ===

=== INCAR ===
EDIFF = 0.00010000

=== PARAMETERS ===
EDIFF = 0.00010000
IALGO = 38
NELM = 60
NELMDL = -5
NELMIN = 2
NELM = 1

=== Electronic Step Energy Changes ===
SCF Step 1: {'alphaZ': '36.76103195', 'ewald': '-1905.04210514', 'hartreedc': '-1174.91276111', 'XCdc': '55.80902972', 'pawpsdc': '2441.71844276', 'pawaedc': '-2404.81769915', 'eentropy': '-0.00000000', 'bandstr': '-59.09445218', 'atom': '3352.92657294', 'e_fr_energy': '343.34805980', 'e_wo_entrp': '343.34805980', 'e_0_energy': '343.34805980'}
SCF Step 2: {'e_fr_energy': '-2.41111302', 'e_wo_entrp': '-2.41111302', 'e_0_energy': '-2.41111302'}
SCF Step 3: {'e_fr_energy': '-45.44904364', 'e_wo_entrp': '-45.44904364', 'e_0_energy': '-45.44904364'}
SCF Step 4: {'e_fr_energy': '-47.18911932', 'e_wo_entrp': '-47.18911932', 'e_0_energy': '-47.18911932'}
SCF Step 5: {'e_fr_energy': '-47.23802052', 'e_wo_entrp': '-47.23802052', 'e_0_energy': '-47.23802052'}
SCF Step 6: {'e_fr_energy': '-44.139879

In [ ]:
# =====================================================
# Step 28: Identify NELM XML Parameter Locations
# =====================================================

print("=== NELM XML Location Analysis ===")

def find_parameter_paths(element, path=""):

    current_path = f"{path}/{element.tag}"

    if "name" in element.attrib:
        current_path += f"[@name='{element.attrib['name']}']"

    if element.attrib.get("name") in [
        "NELM",
        "NELMIN",
        "NELMDL",
        "IALGO"
    ]:
        print(
            f"{current_path} = "
            f"{element.text.strip() if element.text else None}"
        )

    for child in element:
        find_parameter_paths(child, current_path)

parameters = root.find("parameters")

if parameters is not None:
    find_parameter_paths(parameters, "/modeling")
else:
    print("Parameters section not found")

=== NELM XML Location Analysis ===
/modeling/parameters/separator[@name='electronic']/i[@name='IALGO'] = 38
/modeling/parameters/separator[@name='electronic']/separator[@name='electronic convergence']/i[@name='NELM'] = 60
/modeling/parameters/separator[@name='electronic']/separator[@name='electronic convergence']/i[@name='NELMDL'] = -5
/modeling/parameters/separator[@name='electronic']/separator[@name='electronic convergence']/i[@name='NELMIN'] = 2
/modeling/parameters/separator[@name='response functions']/i[@name='NELM'] = 1


In [ ]:
# =====================================================
# Step 29: Batch MASnI3 Relaxation DFT Quality Check
# =====================================================

import requests
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np
import time

API_URL = "https://nomad-lab.eu/prod/v1/api/v1"

# 六份不同的 Relaxation XML
relax_files = classified_df[
    classified_df["calculation_folder"] == "relax"
][["entry_id", "upload_id", "mainfile"]].drop_duplicates()

results = []

session = requests.Session()

def get_value(section, name):
    if section is None:
        return None
    element = section.find(f".//i[@name='{name}']")
    return element.text.strip() if element is not None and element.text else None

for _, row in relax_files.iterrows():

    url = (
        f"{API_URL}/uploads/"
        f"{row['upload_id']}/raw/{row['mainfile']}"
    )

    response = session.get(url, timeout=120)
    response.raise_for_status()

    xml_root = ET.fromstring(response.content)

    incar = xml_root.find("incar")
    params = xml_root.find("parameters")

    electronic = params.find(
        "./separator[@name='electronic']"
    )

    convergence = electronic.find(
        "./separator[@name='electronic convergence']"
    ) if electronic is not None else None

    calculations = xml_root.findall("calculation")

    last_calc = calculations[-1]

    scsteps = last_calc.findall("scstep")

    energies = []

    for step in scsteps:
        value = step.find(
            "./energy/i[@name='e_fr_energy']"
        )

        if value is not None and value.text:
            energies.append(float(value.text))

    delta_e = (
        abs(energies[-1] - energies[-2])
        if len(energies) >= 2
        else np.nan
    )

    ediff_text = get_value(incar, "EDIFF") or get_value(convergence, "EDIFF")
    ediff = float(ediff_text) if ediff_text is not None else np.nan

    results.append({
        "entry_id": row["entry_id"],
        "GGA": get_value(incar, "GGA"),
        "LUSE_VDW": get_value(incar, "LUSE_VDW"),
        "AGGAC": get_value(incar, "AGGAC"),
        "ENCUT": get_value(incar, "ENCUT"),
        "NELM": get_value(convergence, "NELM"),
        "EDIFF": ediff,
        "ionic_steps": len(calculations),
        "last_scf_steps": len(scsteps),
        "last_delta_E_eV": delta_e,
        "energy_threshold_pass": (
            bool(delta_e < ediff)
            if np.isfinite(delta_e) and np.isfinite(ediff)
            else None
        )
    })

    print("Processed:", row["entry_id"])

    time.sleep(2)

quality_df = pd.DataFrame(results)

print("\n=== MASnI3 DFT Batch Quality Summary ===")

print(quality_df.to_string(index=False))

print("\n=== Energy Threshold Check ===")
print(
    quality_df["energy_threshold_pass"].value_counts(
        dropna=False
    )
)

quality_df.to_csv(
    "/content/MASnI3_DFT_quality_check.csv",
    index=False
)

print("\nSaved: MASnI3_DFT_quality_check.csv")

Processed: 0hlSL9bz6yPH2CmO-NRbrr_vK8U6
Processed: 4WF5u7xlskuyKBUyEn9AM2VT81fV
Processed: ETs3KhaeW_GioQu8DvV_r_TdXXe4
Processed: Kgu_QRs28ZBoNYRCL6xgk5HRzkkc
Processed: HrYiscR1-QjcNQmXsKhc1m3tE8KV
Processed: spCTVHlkIR5DAzJZii934icHZwrF

=== MASnI3 DFT Batch Quality Summary ===
                    entry_id GGA LUSE_VDW      AGGAC        ENCUT NELM  EDIFF  ionic_steps  last_scf_steps  last_delta_E_eV  energy_threshold_pass
0hlSL9bz6yPH2CmO-NRbrr_vK8U6  ML        T 0.00000000 400.00000000   60 0.0001            1              11         0.000013                   True
4WF5u7xlskuyKBUyEn9AM2VT81fV  ML        T 0.00000000 400.00000000   60 0.0001            1              13         0.000004                   True
ETs3KhaeW_GioQu8DvV_r_TdXXe4  ML        T 0.00000000 400.00000000   60 0.0001            3               2         0.000002                   True
Kgu_QRs28ZBoNYRCL6xgk5HRzkkc  ML        T 0.00000000 400.00000000   60 0.0001            1              12         0.000024       

In [ ]:
# =====================================================
# Step 30: Check Ionic Forces in VASP Relaxation
# =====================================================

import requests
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import time

records = []

session = requests.Session()

for _, row in relax_files.iterrows():

    url = (
        f"{API_URL}/uploads/"
        f"{row['upload_id']}/raw/{row['mainfile']}"
    )

    response = session.get(url, timeout=120)
    response.raise_for_status()

    xml_root = ET.fromstring(response.content)

    calculations = xml_root.findall("calculation")

    for step_index, calc in enumerate(calculations):

        forces_section = calc.find(
            "./varray[@name='forces']"
        )

        if forces_section is None:
            continue

        forces = np.array([
            [float(x) for x in v.text.split()]
            for v in forces_section.findall("v")
        ])

        magnitudes = np.linalg.norm(forces, axis=1)
        max_force = float(np.max(magnitudes))

        records.append({
            "entry_id": row["entry_id"],
            "ionic_step": step_index + 1,
            "total_ionic_steps": len(calculations),
            "max_force_eV_A": max_force,
            "is_final_step": (
                step_index == len(calculations) - 1
            ),
            "force_below_0.01": max_force < 0.01
        })

    print("Processed:", row["entry_id"])
    time.sleep(2)

ionic_quality_df = pd.DataFrame(records)

print("\n=== MASnI3 Ionic Force Verification ===")

print(ionic_quality_df.to_string(index=False))

print("\n=== Final Ionic Steps ===")

final_steps = ionic_quality_df[
    ionic_quality_df["is_final_step"]
]

print(
    final_steps[
        [
            "entry_id",
            "max_force_eV_A",
            "force_below_0.01"
        ]
    ].to_string(index=False)
)

print("\n=== Summary ===")
print("Total configurations:", len(ionic_quality_df))
print("Final ionic structures:", len(final_steps))
print(
    "Final structures below force threshold:",
    int(final_steps["force_below_0.01"].sum())
)

Processed: 0hlSL9bz6yPH2CmO-NRbrr_vK8U6
Processed: 4WF5u7xlskuyKBUyEn9AM2VT81fV
Processed: ETs3KhaeW_GioQu8DvV_r_TdXXe4
Processed: Kgu_QRs28ZBoNYRCL6xgk5HRzkkc
Processed: HrYiscR1-QjcNQmXsKhc1m3tE8KV
Processed: spCTVHlkIR5DAzJZii934icHZwrF

=== MASnI3 Ionic Force Verification ===
                    entry_id  ionic_step  total_ionic_steps  max_force_eV_A  is_final_step  force_below_0.01
0hlSL9bz6yPH2CmO-NRbrr_vK8U6           1                  1        0.008390           True              True
4WF5u7xlskuyKBUyEn9AM2VT81fV           1                  1        0.007971           True              True
ETs3KhaeW_GioQu8DvV_r_TdXXe4           1                  3        0.013564          False             False
ETs3KhaeW_GioQu8DvV_r_TdXXe4           2                  3        0.016697          False             False
ETs3KhaeW_GioQu8DvV_r_TdXXe4           3                  3        0.009126           True              True
Kgu_QRs28ZBoNYRCL6xgk5HRzkkc           1                  1      

In [ ]:
# =====================================================
# Step 31.1: Build MASnI3 DFT EFS Dataset
# =====================================================

import numpy as np
import pandas as pd
from pymatgen.core import Structure

EV_TO_J = 1.602176634e-19
ANGSTROM_TO_M = 1e-10

dft_dataset = []

for _, row in relax_df.iterrows():

    entry_id = row["entry_id"]
    run_index = int(row["run_index"])
    calc_index = int(row["calc_index"])

    archive = vasp_archives[entry_id]
    run = archive["run"][run_index]
    calc = run["calculation"][calc_index]

    system_index = int(
        calc["system_ref"].split("/")[-1]
    )

    atoms = run["system"][system_index]["atoms"]

    lattice = (
        np.array(atoms["lattice_vectors"], dtype=float)
        / ANGSTROM_TO_M
    )

    positions = (
        np.array(atoms["positions"], dtype=float)
        / ANGSTROM_TO_M
    )

    structure = Structure(
        lattice=lattice,
        species=atoms["labels"],
        coords=positions,
        coords_are_cartesian=True
    )

    energy = (
        float(calc["energy"]["total"]["value"])
        / EV_TO_J
    )

    forces = (
        np.array(
            calc["forces"]["total"]["value"],
            dtype=float
        )
        * ANGSTROM_TO_M
        / EV_TO_J
    )

    stress = (
        np.array(
            calc["stress"]["total"]["value"],
            dtype=float
        )
        / 1e9
    )

    assert forces.shape == (len(structure), 3)
    assert stress.shape == (3, 3)
    assert np.isfinite(energy)
    assert np.all(np.isfinite(forces))
    assert np.all(np.isfinite(stress))

    dft_dataset.append({
        "entry_id": entry_id,
        "run_index": run_index,
        "calc_index": calc_index,
        "structure": structure,
        "energy": energy,
        "forces": forces,
        "stress": stress
    })

print("=== MASnI3 DFT EFS Dataset ===")

print("Total configurations:", len(dft_dataset))

print("\n=== First Configuration ===")

sample = dft_dataset[0]

print("Entry ID:", sample["entry_id"])
print("Formula:", sample["structure"].formula)
print("Atoms:", len(sample["structure"]))
print("Total energy:", sample["energy"], "eV")
print("Forces shape:", sample["forces"].shape)
print("Stress shape:", sample["stress"].shape)

print("\nDataset construction completed!")

=== MASnI3 DFT EFS Dataset ===
Total configurations: 12

=== First Configuration ===
Entry ID: 0hlSL9bz6yPH2CmO-NRbrr_vK8U6
Formula: Sn1 H6 C1 I3 N1
Atoms: 12
Total energy: -43.68691386 eV
Forces shape: (12, 3)
Stress shape: (3, 3)

Dataset construction completed!


# 將 DFT 資料集永久儲存至 Google Drive

In [ ]:
# =====================================================
# Step 31.2: Save MASnI3 DFT Dataset to Google Drive
# =====================================================

from google.colab import drive
from pathlib import Path
import json

drive.mount("/content/drive")

SAVE_DIR = Path(
    "/content/drive/MyDrive/MASnI3_DFT_Dataset"
)

SAVE_DIR.mkdir(parents=True, exist_ok=True)

dataset_export = []

for item in dft_dataset:

    dataset_export.append({
        "entry_id": item["entry_id"],
        "run_index": item["run_index"],
        "calc_index": item["calc_index"],
        "structure": item["structure"].as_dict(),
        "energy_eV": float(item["energy"]),
        "forces_eV_A": item["forces"].tolist(),
        "stress_GPa": item["stress"].tolist()
    })

save_path = SAVE_DIR / "MASnI3_DFT_EFS_dataset.json"

with open(save_path, "w") as f:
    json.dump(dataset_export, f, indent=2)

print("=== MASnI3 DFT Dataset Saved ===")
print("File:", save_path)
print("Configurations:", len(dataset_export))
print("File exists:", save_path.exists())
print("File size:", save_path.stat().st_size, "bytes")

Mounted at /content/drive
=== MASnI3 DFT Dataset Saved ===
File: /content/drive/MyDrive/MASnI3_DFT_Dataset/MASnI3_DFT_EFS_dataset.json
Configurations: 12
File exists: True
File size: 90231 bytes


# **重新啟動Colab>>執行>>叫出JSON檔案**

In [7]:
from google.colab import drive
from pymatgen.core import Structure
import numpy as np
import json

drive.mount("/content/drive")

dataset_path = (
    "/content/drive/MyDrive/"
    "MASnI3_DFT_Dataset/MASnI3_DFT_EFS_dataset.json"
)

with open(dataset_path, "r") as f:
    saved_data = json.load(f)

dft_dataset = []

for item in saved_data:
    dft_dataset.append({
        "entry_id": item["entry_id"],
        "run_index": item["run_index"],
        "calc_index": item["calc_index"],
        "structure": Structure.from_dict(item["structure"]),
        "energy": item["energy_eV"],
        "forces": np.array(item["forces_eV_A"]),
        "stress": np.array(item["stress_GPa"])
    })

print("MASnI3 DFT dataset loaded successfully!")
print("Configurations:", len(dft_dataset))

Mounted at /content/drive
MASnI3 DFT dataset loaded successfully!
Configurations: 12


In [8]:
# =====================================================
# Step 32: MASnI3 DFT Dataset Validation
# =====================================================

import numpy as np
from collections import Counter

print("=== MASnI3 DFT Dataset Validation ===")

valid_count = 0
invalid_count = 0

for i, item in enumerate(dft_dataset):

    structure = item["structure"]
    energy = item["energy"]
    forces = np.asarray(item["forces"])
    stress = np.asarray(item["stress"])

    valid = (
        len(structure) == 12
        and dict(Counter(str(site.specie) for site in structure)) ==
        {"Sn": 1, "H": 6, "C": 1, "I": 3, "N": 1}
        and np.isfinite(energy)
        and forces.shape == (12, 3)
        and stress.shape == (3, 3)
        and np.all(np.isfinite(forces))
        and np.all(np.isfinite(stress))
        and structure.volume > 0
    )

    if valid:
        valid_count += 1
    else:
        invalid_count += 1
        print(f"Invalid configuration: {i}")

print("Total configurations:", len(dft_dataset))
print("Valid configurations:", valid_count)
print("Invalid configurations:", invalid_count)

print("\n=== DFT Energy Range ===")

energies = [
    item["energy"] / len(item["structure"])
    for item in dft_dataset
]

print("Minimum:", min(energies), "eV/atom")
print("Maximum:", max(energies), "eV/atom")

print("\n=== Validation Result ===")

if valid_count == 12 and invalid_count == 0:
    print("All MASnI3 DFT configurations passed!")
else:
    print("Dataset requires further inspection.")

=== MASnI3 DFT Dataset Validation ===
Total configurations: 12
Valid configurations: 12
Invalid configurations: 0

=== DFT Energy Range ===
Minimum: -3.6411973275 eV/atom
Maximum: -3.6384424716666666 eV/atom

=== Validation Result ===
All MASnI3 DFT configurations passed!


In [ ]:
# =====================================================
# Step 33: Verify NOMAD Stress Against VASP XML
# =====================================================

import numpy as np
import requests
import xml.etree.ElementTree as ET

sample = dft_dataset[0]

entry_id = sample["entry_id"]
calc_index = int(sample["calc_index"])

source = classified_df[
    classified_df["entry_id"] == entry_id
].iloc[0]

url = (
    "https://nomad-lab.eu/prod/v1/api/v1/"
    f"uploads/{source['upload_id']}/raw/{source['mainfile']}"
)

response = requests.get(url, timeout=120)
response.raise_for_status()

xml_root = ET.fromstring(response.content)

calculations = xml_root.findall("calculation")
calc = calculations[calc_index]

stress_section = calc.find("./varray[@name='stress']")

if stress_section is None:
    raise ValueError("Stress tensor not found in VASP XML")

# VASP XML stress: kbar
vasp_stress_kbar = np.array([
    [float(x) for x in v.text.split()]
    for v in stress_section.findall("v")
])

# 1 kbar = 0.1 GPa
vasp_stress_GPa = vasp_stress_kbar * 0.1

# 先前由 NOMAD Archive 轉換並保存的 Stress
nomad_stress_GPa = np.asarray(
    sample["stress"],
    dtype=float
)

print("=== VASP Original Stress (GPa) ===")
print(vasp_stress_GPa)

print("\n=== NOMAD Dataset Stress (GPa) ===")
print(nomad_stress_GPa)

same_sign_error = np.max(
    np.abs(vasp_stress_GPa - nomad_stress_GPa)
)

opposite_sign_error = np.max(
    np.abs(vasp_stress_GPa + nomad_stress_GPa)
)

print("\n=== Stress Comparison ===")
print("Same-sign maximum difference:", same_sign_error)
print("Opposite-sign maximum difference:", opposite_sign_error)

if same_sign_error < 1e-5:
    print("Result: NOMAD and VASP stress use the same sign.")
elif opposite_sign_error < 1e-5:
    print("Result: NOMAD and VASP stress use opposite signs.")
else:
    print("Result: Stress mismatch requires further inspection.")

=== VASP Original Stress (GPa) ===
[[ 0.01077328 -0.00104917 -0.00212196]
 [-0.00104917  0.01047269 -0.00052745]
 [-0.00212196 -0.00052745  0.01097879]]

=== NOMAD Dataset Stress (GPa) ===
[[ 0.01077328 -0.00104917 -0.00212196]
 [-0.00104917  0.01047269 -0.00052745]
 [-0.00212196 -0.00052745  0.01097879]]

=== Stress Comparison ===
Same-sign maximum difference: 1.734723475976807e-18
Opposite-sign maximum difference: 0.021957578
Result: NOMAD and VASP stress use the same sign.


In [ ]:
# =====================================================
# Step 34: CHGNet Baseline - First MASnI3 Structure
# =====================================================

import numpy as np
from chgnet.model.model import CHGNet

# 載入官方預訓練 CHGNet
chgnet = CHGNet.load()

# 取得第一筆 DFT 資料
sample = dft_dataset[0]

structure = sample["structure"]

# 使用原始 CHGNet 預測
prediction = chgnet.predict_structure(structure)

# CHGNet 預測結果
pred_energy = float(prediction["e"])  # eV/atom
pred_forces = np.asarray(prediction["f"])
pred_stress = np.asarray(prediction["s"])

# NOMAD DFT 參考值
dft_energy = sample["energy"] / len(structure)
dft_forces = np.asarray(sample["forces"])
dft_stress = np.asarray(sample["stress"])

print("=== MASnI3 CHGNet Baseline ===")

print("\n=== Energy (eV/atom) ===")
print("DFT:", dft_energy)
print("CHGNet:", pred_energy)
print("Absolute error:", abs(pred_energy - dft_energy))

print("\n=== Forces (eV/Å) ===")
print("DFT shape:", dft_forces.shape)
print("CHGNet shape:", pred_forces.shape)

force_mae = np.mean(
    np.abs(pred_forces - dft_forces)
)

print("Force MAE:", force_mae)

print("\n=== Stress (GPa) ===")
print("DFT shape:", dft_stress.shape)
print("CHGNet shape:", pred_stress.shape)

stress_mae = np.mean(
    np.abs(pred_stress - dft_stress)
)

print("Stress MAE:", stress_mae)

print("\nBaseline prediction completed!")

CHGNet v0.3.0 initialized with 412,525 parameters
CHGNet will run on cpu


/usr/local/lib/python3.13/dist-packages/chgnet/model/model.py:898: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  volumes = torch.tensor(volumes, dtype=TORCH_DTYPE, device=atomic_numbers.device)


=== MASnI3 CHGNet Baseline ===

=== Energy (eV/atom) ===
DFT: -3.6405761549999998
CHGNet: -4.365141868591309
Absolute error: 0.7245657135913088

=== Forces (eV/Å) ===
DFT shape: (12, 3)
CHGNet shape: (12, 3)
Force MAE: 0.06126167084666492

=== Stress (GPa) ===
DFT shape: (3, 3)
CHGNet shape: (3, 3)
Stress MAE: 0.169924186287976

Baseline prediction completed!


In [ ]:
# =====================================================
# Step 35: CHGNet Baseline Evaluation - All 12 MASnI3
# =====================================================

import numpy as np
import pandas as pd

baseline_results = []

for i, sample in enumerate(dft_dataset):

    structure = sample["structure"]

    # CHGNet prediction
    prediction = chgnet.predict_structure(structure)

    pred_energy = float(prediction["e"])
    pred_forces = np.asarray(prediction["f"])
    pred_stress = np.asarray(prediction["s"])

    # DFT reference
    dft_energy = float(sample["energy"]) / len(structure)
    dft_forces = np.asarray(sample["forces"])
    dft_stress = np.asarray(sample["stress"])

    # Validate predicted shapes
    assert pred_forces.shape == dft_forces.shape
    assert pred_stress.shape == dft_stress.shape

    baseline_results.append({
        "entry_id": sample["entry_id"],
        "calc_index": sample["calc_index"],
        "dft_energy_eV_atom": dft_energy,
        "chgnet_energy_eV_atom": pred_energy,
        "energy_error_eV_atom": pred_energy - dft_energy,
        "energy_abs_error_eV_atom": abs(pred_energy - dft_energy),
        "force_mae_eV_A": np.mean(
            np.abs(pred_forces - dft_forces)
        ),
        "stress_mae_GPa": np.mean(
            np.abs(pred_stress - dft_stress)
        )
    })

    print(f"Processed: {i + 1}/{len(dft_dataset)}")

baseline_df = pd.DataFrame(baseline_results)

print("\n=== MASnI3 CHGNet Baseline Results ===")

print(
    baseline_df[
        [
            "entry_id",
            "calc_index",
            "energy_abs_error_eV_atom",
            "force_mae_eV_A",
            "stress_mae_GPa"
        ]
    ].to_string(index=False)
)

print("\n=== Overall Baseline MAE ===")

print(
    "Energy MAE:",
    baseline_df["energy_abs_error_eV_atom"].mean(),
    "eV/atom"
)

print(
    "Force MAE:",
    baseline_df["force_mae_eV_A"].mean(),
    "eV/Å"
)

print(
    "Stress MAE:",
    baseline_df["stress_mae_GPa"].mean(),
    "GPa"
)

# Save baseline results
baseline_df.to_csv(
    "/content/MASnI3_CHGNet_baseline.csv",
    index=False
)

print("\nSaved: MASnI3_CHGNet_baseline.csv")

Processed: 1/12
Processed: 2/12
Processed: 3/12
Processed: 4/12
Processed: 5/12
Processed: 6/12
Processed: 7/12
Processed: 8/12
Processed: 9/12
Processed: 10/12
Processed: 11/12
Processed: 12/12

=== MASnI3 CHGNet Baseline Results ===
                    entry_id  calc_index  energy_abs_error_eV_atom  force_mae_eV_A  stress_mae_GPa
0hlSL9bz6yPH2CmO-NRbrr_vK8U6           0                  0.724566        0.061262        0.169924
4WF5u7xlskuyKBUyEn9AM2VT81fV           0                  0.716492        0.127130        0.299772
ETs3KhaeW_GioQu8DvV_r_TdXXe4           0                  0.724489        0.077437        0.181147
ETs3KhaeW_GioQu8DvV_r_TdXXe4           1                  0.724478        0.077489        0.187993
ETs3KhaeW_GioQu8DvV_r_TdXXe4           2                  0.724490        0.077255        0.186354
Kgu_QRs28ZBoNYRCL6xgk5HRzkkc           0                  0.717118        0.116641        0.283808
HrYiscR1-QjcNQmXsKhc1m3tE8KV           0                  0.713648      

In [ ]:
# =====================================================
# Step 36: MASnI3 Energy Offset Analysis
# =====================================================

import numpy as np

# Signed energy error
errors = (
    baseline_df["chgnet_energy_eV_atom"]
    - baseline_df["dft_energy_eV_atom"]
).to_numpy()

# Mean systematic offset
mean_offset = np.mean(errors)

# Diagnostic residuals after removing the mean offset
residuals = errors - mean_offset

print("=== MASnI3 Energy Offset Analysis ===")

print("\n=== Original Energy Errors ===")
print("Mean signed error:", mean_offset, "eV/atom")
print("Original MAE:", np.mean(np.abs(errors)), "eV/atom")

print("\n=== Offset-Corrected Residuals ===")
print("Residual MAE:", np.mean(np.abs(residuals)), "eV/atom")
print("Residual RMSE:", np.sqrt(np.mean(residuals**2)), "eV/atom")
print("Maximum residual:", np.max(np.abs(residuals)), "eV/atom")

print("\n=== Energy Error Distribution ===")

for i, error in enumerate(errors, 1):
    print(
        f"Configuration {i:2d}: "
        f"Original = {error:+.6f}, "
        f"Residual = {residuals[i-1]:+.6f} eV/atom"
    )

=== MASnI3 Energy Offset Analysis ===

=== Original Energy Errors ===
Mean signed error: -0.7186301429896038 eV/atom
Original MAE: 0.7186301429896038 eV/atom

=== Offset-Corrected Residuals ===
Residual MAE: 0.004771475605926538 eV/atom
Residual RMSE: 0.004966749753857852 eV/atom
Maximum residual: 0.00593557060170502 eV/atom

=== Energy Error Distribution ===
Configuration  1: Original = -0.724566, Residual = -0.005936 eV/atom
Configuration  2: Original = -0.716492, Residual = +0.002138 eV/atom
Configuration  3: Original = -0.724489, Residual = -0.005859 eV/atom
Configuration  4: Original = -0.724478, Residual = -0.005848 eV/atom
Configuration  5: Original = -0.724490, Residual = -0.005860 eV/atom
Configuration  6: Original = -0.717118, Residual = +0.001512 eV/atom
Configuration  7: Original = -0.713648, Residual = +0.004982 eV/atom
Configuration  8: Original = -0.713627, Residual = +0.005003 eV/atom
Configuration  9: Original = -0.713628, Residual = +0.005003 eV/atom
Configuration 10:

# 儲存檔案至雲端

In [ ]:
# =====================================================
# Step 35.1: Save MASnI3 Research Checkpoint
# =====================================================

from pathlib import Path
import pandas as pd
import json

SAVE_DIR = Path(
    "/content/drive/MyDrive/MASnI3_DFT_Dataset"
)

SAVE_DIR.mkdir(parents=True, exist_ok=True)

# 1. 保存 CHGNet Baseline
baseline_path = SAVE_DIR / "MASnI3_CHGNet_baseline.csv"

baseline_df.to_csv(
    baseline_path,
    index=False
)

# 2. 保存 DFT 品質檢查結果
quality_path = SAVE_DIR / "MASnI3_DFT_quality_check.csv"

if "quality_df" in globals():
    quality_df.to_csv(
        quality_path,
        index=False
    )

# 3. 確認 DFT EFS 資料集
dataset_path = SAVE_DIR / "MASnI3_DFT_EFS_dataset.json"

# 4. 保存研究進度資訊
checkpoint = {
    "material": "MASnI3",
    "completed_step": 35,
    "next_step": 36,
    "dft_configurations": 12,
    "baseline_energy_mae_eV_atom": float(
        baseline_df["energy_abs_error_eV_atom"].mean()
    ),
    "baseline_force_mae_eV_A": float(
        baseline_df["force_mae_eV_A"].mean()
    ),
    "baseline_stress_mae_GPa": float(
        baseline_df["stress_mae_GPa"].mean()
    )
}

checkpoint_path = SAVE_DIR / "MASnI3_checkpoint.json"

with open(checkpoint_path, "w") as f:
    json.dump(checkpoint, f, indent=2)

# 5. 驗證儲存結果
print("=== MASnI3 Research Checkpoint ===")

for path in [
    dataset_path,
    baseline_path,
    quality_path,
    checkpoint_path
]:
    print(
        f"{'✓' if path.exists() else '✗'} "
        f"{path.name}"
    )

print("\n=== Current Progress ===")
print("Completed Step:", checkpoint["completed_step"])
print("Next Step:", checkpoint["next_step"])
print("DFT configurations:", checkpoint["dft_configurations"])

print("\nCheckpoint saved successfully!")

=== MASnI3 Research Checkpoint ===
✓ MASnI3_DFT_EFS_dataset.json
✓ MASnI3_CHGNet_baseline.csv
✓ MASnI3_DFT_quality_check.csv
✓ MASnI3_checkpoint.json

=== Current Progress ===
Completed Step: 35
Next Step: 36
DFT configurations: 12

Checkpoint saved successfully!


# 重新執行>>載入JSON/CSV

In [9]:
# =====================================================
# Restore MASnI3 DFT Dataset and CHGNet Baseline
# =====================================================

from google.colab import drive
from pymatgen.core import Structure
from pathlib import Path
import pandas as pd
import numpy as np
import json

# 1. 掛載 Google Drive
drive.mount("/content/drive")

# 2. 資料儲存位置
DATA_DIR = Path(
    "/content/drive/MyDrive/MASnI3_DFT_Dataset"
)

# 3. 載入 DFT EFS Dataset
with open(DATA_DIR / "MASnI3_DFT_EFS_dataset.json", "r") as f:
    saved_data = json.load(f)

dft_dataset = []

for item in saved_data:
    dft_dataset.append({
        "entry_id": item["entry_id"],
        "run_index": item["run_index"],
        "calc_index": item["calc_index"],
        "structure": Structure.from_dict(item["structure"]),
        "energy": float(item["energy_eV"]),
        "forces": np.array(item["forces_eV_A"]),
        "stress": np.array(item["stress_GPa"])
    })

# 4. 載入 CHGNet Baseline
baseline_df = pd.read_csv(
    DATA_DIR / "MASnI3_CHGNet_baseline.csv"
)

# 5. 載入 DFT 品質檢查結果
quality_df = pd.read_csv(
    DATA_DIR / "MASnI3_DFT_quality_check.csv"
)

# 6. 載入研究進度
with open(DATA_DIR / "MASnI3_checkpoint.json", "r") as f:
    checkpoint = json.load(f)

# 7. 確認資料恢復成功
print("=== MASnI3 Research Data Recovery ===")

print("DFT configurations:", len(dft_dataset))
print("CHGNet Baseline records:", len(baseline_df))
print("DFT quality records:", len(quality_df))

print("\n=== Research Progress ===")
print("Completed Step:", checkpoint["completed_step"])
print("Next Step:", checkpoint["next_step"])

print("\nAll MASnI3 research data loaded successfully!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== MASnI3 Research Data Recovery ===
DFT configurations: 12
CHGNet Baseline records: 12
DFT quality records: 6

=== Research Progress ===
Completed Step: 35
Next Step: 36

All MASnI3 research data loaded successfully!


In [10]:
# =====================================================
# Step 37: MASnI3 Relative Energy Benchmark
# =====================================================

import numpy as np
from scipy.stats import pearsonr, spearmanr

# DFT and CHGNet energy (eV/atom)
dft_e = baseline_df["dft_energy_eV_atom"].to_numpy()
chgnet_e = baseline_df["chgnet_energy_eV_atom"].to_numpy()

# Remove the mean energy to analyze relative variations
dft_relative = dft_e - np.mean(dft_e)
chgnet_relative = chgnet_e - np.mean(chgnet_e)

# CHGNet relative-energy error
chgnet_relative_mae = np.mean(
    np.abs(chgnet_relative - dft_relative)
)

# Constant mean-energy benchmark
mean_baseline_mae = np.mean(
    np.abs(dft_relative)
)

# Energy correlations
pearson_r = pearsonr(dft_e, chgnet_e).statistic
spearman_r = spearmanr(dft_e, chgnet_e).statistic

print("=== MASnI3 Relative Energy Benchmark ===")

print("\n=== Energy Variation ===")
print(
    "DFT energy range:",
    np.ptp(dft_e) * 1000,
    "meV/atom"
)

print(
    "CHGNet energy range:",
    np.ptp(chgnet_e) * 1000,
    "meV/atom"
)

print("\n=== Relative Energy MAE ===")
print(
    "CHGNet:",
    chgnet_relative_mae * 1000,
    "meV/atom"
)

print(
    "Mean-energy baseline:",
    mean_baseline_mae * 1000,
    "meV/atom"
)

print("\n=== Energy Correlation ===")
print("Pearson r:", pearson_r)
print("Spearman r:", spearman_r)

print("\n=== Benchmark Result ===")

if chgnet_relative_mae < mean_baseline_mae:
    print("CHGNet outperforms the constant-energy baseline.")
else:
    print("CHGNet does not outperform the constant-energy baseline.")

=== MASnI3 Relative Energy Benchmark ===

=== Energy Variation ===
DFT energy range: 2.7548558333334583 meV/atom
CHGNet energy range: 10.74075698852539 meV/atom

=== Relative Energy MAE ===
CHGNet: 4.771475605926279 meV/atom
Mean-energy baseline: 0.4542390277777475 meV/atom

=== Energy Correlation ===
Pearson r: -0.27008513083738994
Spearman r: -0.5244755244755246

=== Benchmark Result ===
CHGNet does not outperform the constant-energy baseline.


In [11]:
# =====================================================
# Step 38: MASnI3 Fine-tuning Data Group Analysis
# =====================================================

import pandas as pd
import numpy as np

# 使用已載入的 DFT EFS 資料集
rows = []

for item in dft_dataset:

    structure = item["structure"]
    energy_per_atom = (
        float(item["energy"]) / len(structure)
    )

    rows.append({
        "entry_id": item["entry_id"],
        "run_index": item["run_index"],
        "calc_index": item["calc_index"],
        "n_atoms": len(structure),
        "energy_eV_atom": energy_per_atom
    })

training_df = pd.DataFrame(rows)

# 依照 VASP Entry ID 分組
group_summary = training_df.groupby(
    "entry_id"
).agg(
    configurations=("energy_eV_atom", "count"),
    minimum_energy=("energy_eV_atom", "min"),
    maximum_energy=("energy_eV_atom", "max")
)

# 計算各組內的能量變化
group_summary["energy_range_meV_atom"] = (
    group_summary["maximum_energy"]
    - group_summary["minimum_energy"]
) * 1000

print("=== MASnI3 Fine-tuning Group Analysis ===")

print("\nTotal configurations:", len(training_df))
print("Independent Entry IDs:",
      training_df["entry_id"].nunique())

print("\n=== Group Distribution ===")
print(group_summary.to_string())

print("\n=== Overall Energy Statistics ===")

print(
    "Energy standard deviation:",
    training_df["energy_eV_atom"].std(),
    "eV/atom"
)

print(
    "Overall energy range:",
    (
        training_df["energy_eV_atom"].max()
        - training_df["energy_eV_atom"].min()
    ) * 1000,
    "meV/atom"
)

print("\n=== Data Splitting Recommendation ===")

print(
    "Split by Entry ID to avoid "
    "relaxation-trajectory leakage."
)

print(
    "Recommended evaluation: "
    "Leave-One-Entry-ID-Out Cross-Validation."
)

=== MASnI3 Fine-tuning Group Analysis ===

Total configurations: 12
Independent Entry IDs: 6

=== Group Distribution ===
                              configurations  minimum_energy  maximum_energy  energy_range_meV_atom
entry_id                                                                                           
0hlSL9bz6yPH2CmO-NRbrr_vK8U6               1       -3.640576       -3.640576               0.000000
4WF5u7xlskuyKBUyEn9AM2VT81fV               1       -3.639720       -3.639720               0.000000
ETs3KhaeW_GioQu8DvV_r_TdXXe4               3       -3.640507       -3.640506               0.000585
HrYiscR1-QjcNQmXsKhc1m3tE8KV               5       -3.640777       -3.640772               0.004735
Kgu_QRs28ZBoNYRCL6xgk5HRzkkc               1       -3.641197       -3.641197               0.000000
spCTVHlkIR5DAzJZii934icHZwrF               1       -3.638442       -3.638442               0.000000

=== Overall Energy Statistics ===
Energy standard deviation: 0.000718146120281

In [12]:
# =====================================================
# Step 39: MASnI3 Leave-One-Entry-ID-Out Splitting
# =====================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import LeaveOneGroupOut

# 1. 取得資料分組
groups = training_df["entry_id"].values

# 2. 建立交叉驗證
logo = LeaveOneGroupOut()

cv_splits = []

print("=== MASnI3 Leave-One-Entry-ID-Out CV ===")

# 3. 建立每一組訓練與測試資料
for fold, (train_idx, test_idx) in enumerate(
    logo.split(training_df, groups=groups),
    start=1
):

    train_entries = set(
        training_df.iloc[train_idx]["entry_id"]
    )

    test_entries = set(
        training_df.iloc[test_idx]["entry_id"]
    )

    # 確保 Entry ID 沒有重複
    assert train_entries.isdisjoint(test_entries)

    cv_splits.append({
        "fold": fold,
        "train_indices": train_idx.tolist(),
        "test_indices": test_idx.tolist(),
        "test_entry_id": list(test_entries)[0]
    })

    print(f"\nFold {fold}")

    print("Training configurations:", len(train_idx))
    print("Testing configurations:", len(test_idx))

    print("Training Entry IDs:", len(train_entries))
    print("Testing Entry IDs:", len(test_entries))

    print("Test Entry ID:", list(test_entries)[0])

# 4. 最終驗證
all_test_indices = []

for split in cv_splits:
    all_test_indices.extend(split["test_indices"])

assert sorted(all_test_indices) == list(
    range(len(training_df))
)

print("\n=== Cross-Validation Summary ===")

print("Total folds:", len(cv_splits))

print(
    "Total unique test configurations:",
    len(set(all_test_indices))
)

print(
    "Every configuration tested exactly once:",
    len(all_test_indices) == len(training_df)
)

print("\nCross-validation splitting completed!")

=== MASnI3 Leave-One-Entry-ID-Out CV ===

Fold 1
Training configurations: 11
Testing configurations: 1
Training Entry IDs: 5
Testing Entry IDs: 1
Test Entry ID: 0hlSL9bz6yPH2CmO-NRbrr_vK8U6

Fold 2
Training configurations: 11
Testing configurations: 1
Training Entry IDs: 5
Testing Entry IDs: 1
Test Entry ID: 4WF5u7xlskuyKBUyEn9AM2VT81fV

Fold 3
Training configurations: 9
Testing configurations: 3
Training Entry IDs: 5
Testing Entry IDs: 1
Test Entry ID: ETs3KhaeW_GioQu8DvV_r_TdXXe4

Fold 4
Training configurations: 7
Testing configurations: 5
Training Entry IDs: 5
Testing Entry IDs: 1
Test Entry ID: HrYiscR1-QjcNQmXsKhc1m3tE8KV

Fold 5
Training configurations: 11
Testing configurations: 1
Training Entry IDs: 5
Testing Entry IDs: 1
Test Entry ID: Kgu_QRs28ZBoNYRCL6xgk5HRzkkc

Fold 6
Training configurations: 11
Testing configurations: 1
Training Entry IDs: 5
Testing Entry IDs: 1
Test Entry ID: spCTVHlkIR5DAzJZii934icHZwrF

=== Cross-Validation Summary ===
Total folds: 6
Total unique test 

**確認 ZENODO 668 筆中有多少 MASnI₃**

In [13]:
# =====================================================
# Step 39A: Download and Inspect MA(Ge,Sn)I3 DFT Dataset
# =====================================================

import requests
from pathlib import Path
from collections import Counter
from ase.io import iread

# 1. Dataset information
RECORD_ID = "21863902"

DATA_DIR = Path(
    "/content/drive/MyDrive/MASnI3_DFT_Dataset/"
    "External_DFT"
)

DATA_DIR.mkdir(parents=True, exist_ok=True)

# 2. Get Zenodo metadata
api_url = f"https://zenodo.org/api/records/{RECORD_ID}"

response = requests.get(api_url, timeout=30)
response.raise_for_status()

record = response.json()

# 3. Locate train.xyz
files = record.get("files", [])

file_info = next(
    (f for f in files if f["key"] == "train.xyz"),
    None
)

if file_info is None:
    raise FileNotFoundError(
        "train.xyz not found in Zenodo record."
    )

download_url = file_info["links"]["self"]

file_path = DATA_DIR / "MA_Ge_Sn_I3_train.xyz"

# 4. Download once
if not file_path.exists():

    with requests.get(
        download_url,
        stream=True,
        timeout=120
    ) as r:

        r.raise_for_status()

        with open(file_path, "wb") as f:
            for chunk in r.iter_content(
                chunk_size=1024 * 1024
            ):
                if chunk:
                    f.write(chunk)

print("=== Download Result ===")
print("File:", file_path)
print("Size:", file_path.stat().st_size, "bytes")

# 5. Inspect chemical compositions
composition_counter = Counter()
masni3_count = 0
masni3_indices = []
first_atoms = None

for i, atoms in enumerate(iread(str(file_path), index=":")):

    if first_atoms is None:
        first_atoms = atoms.copy()

    counts = Counter(atoms.get_chemical_symbols())

    composition_key = tuple(sorted(counts.items()))
    composition_counter[composition_key] += 1

    # Pure MASnI3: CH6NSnI3 per formula unit
    n = counts.get("Sn", 0)

    is_masni3 = (
        n > 0
        and set(counts) == {"C", "H", "N", "Sn", "I"}
        and counts["C"] == n
        and counts["H"] == 6 * n
        and counts["N"] == n
        and counts["I"] == 3 * n
    )

    if is_masni3:
        masni3_count += 1
        masni3_indices.append(i)

print("\n=== Dataset Composition Analysis ===")

print("Total structures:", sum(composition_counter.values()))
print("Unique compositions:", len(composition_counter))
print("Pure MASnI3 structures:", masni3_count)

print("\n=== Composition Distribution ===")

for composition, count in composition_counter.most_common():
    print(dict(composition), ":", count)

print("\n=== First Structure Properties ===")

if first_atoms is not None:
    print("Formula:", first_atoms.get_chemical_formula())
    print("Atoms:", len(first_atoms))
    print("Info keys:", list(first_atoms.info.keys()))
    print("Array keys:", list(first_atoms.arrays.keys()))
    print("Calculator:", type(first_atoms.calc).__name__
          if first_atoms.calc is not None else "None")

print("\nDataset inspection completed!")

=== Download Result ===
File: /content/drive/MyDrive/MASnI3_DFT_Dataset/External_DFT/MA_Ge_Sn_I3_train.xyz
Size: 6436629 bytes

=== Dataset Composition Analysis ===
Total structures: 668
Unique compositions: 28
Pure MASnI3 structures: 170

=== Composition Distribution ===
{'C': 8, 'H': 48, 'I': 24, 'N': 8, 'Sn': 8} : 151
{'C': 8, 'Ge': 8, 'H': 48, 'I': 24, 'N': 8} : 149
{'C': 8, 'Ge': 4, 'H': 48, 'I': 24, 'N': 8, 'Sn': 4} : 106
{'C': 8, 'Ge': 5, 'H': 48, 'I': 24, 'N': 8, 'Sn': 3} : 28
{'C': 8, 'Ge': 1, 'H': 48, 'I': 24, 'N': 8, 'Sn': 7} : 28
{'C': 8, 'Ge': 7, 'H': 48, 'I': 24, 'N': 8, 'Sn': 1} : 27
{'C': 8, 'Ge': 3, 'H': 48, 'I': 24, 'N': 8, 'Sn': 5} : 26
{'C': 8, 'Ge': 6, 'H': 48, 'I': 24, 'N': 8, 'Sn': 2} : 19
{'C': 4, 'Ge': 2, 'H': 24, 'I': 12, 'N': 4, 'Sn': 2} : 18
{'C': 8, 'Ge': 2, 'H': 48, 'I': 24, 'N': 8, 'Sn': 6} : 17
{'C': 4, 'Ge': 1, 'H': 24, 'I': 12, 'N': 4, 'Sn': 3} : 17
{'C': 4, 'Ge': 3, 'H': 24, 'I': 12, 'N': 4, 'Sn': 1} : 15
{'C': 4, 'H': 24, 'I': 12, 'N': 4, 'Sn': 4} : 

In [14]:
# =====================================================
# Step 39B: Inspect MASnI3 DFT Labels
# =====================================================

from ase.io import iread
from collections import Counter
import numpy as np

file_path = (
    "/content/drive/MyDrive/MASnI3_DFT_Dataset/"
    "External_DFT/MA_Ge_Sn_I3_train.xyz"
)

print("=== Original XYZ Header ===")

with open(file_path, "r") as f:
    print("Line 1:", f.readline().strip())
    print("Line 2:", f.readline().strip()[:1500])
    print("Line 3:", f.readline().strip()[:500])

# Find the first pure MASnI3 structure
target_atoms = None
target_index = None

for i, atoms in enumerate(iread(file_path, index=":")):

    counts = Counter(atoms.get_chemical_symbols())
    n = counts.get("Sn", 0)

    is_masni3 = (
        n > 0
        and set(counts) == {"C", "H", "N", "Sn", "I"}
        and counts["C"] == n
        and counts["H"] == 6 * n
        and counts["N"] == n
        and counts["I"] == 3 * n
    )

    if is_masni3:
        target_atoms = atoms
        target_index = i
        break

assert target_atoms is not None, "No MASnI3 found."

print("\n=== First Pure MASnI3 Structure ===")

print("Structure index:", target_index)
print("Formula:", target_atoms.get_chemical_formula())
print("Atoms:", len(target_atoms))

print("\n=== Available Metadata ===")

for key, value in target_atoms.info.items():
    print(f"{key}: {str(value)[:300]}")

print("\n=== Available Arrays ===")

for key, value in target_atoms.arrays.items():
    print(
        key,
        "Shape:",
        np.asarray(value).shape
    )

print("\n=== DFT Label Availability ===")

print("Calculator:", target_atoms.calc)

for label in ["energy", "forces", "stress", "virial"]:

    in_info = label in target_atoms.info
    in_arrays = label in target_atoms.arrays

    in_calc = (
        target_atoms.calc is not None
        and label in target_atoms.calc.results
    )

    print(
        f"{label}: "
        f"info={in_info}, "
        f"arrays={in_arrays}, "
        f"calculator={in_calc}"
    )

print("\nLabel inspection completed!")

=== Original XYZ Header ===
Line 1: 96
Line 2: Lattice="13.07893978 1.46624788 -0.22700162 -0.34625269 12.84374998 0.24075623 0.50380622 -0.19697301 12.33184246" Properties=species:S:1:pos:R:3:forces:R:3 filename=empty-filename energy=-9.92790425941348 free_energy=-5199068.96790869 pbc="T T T"
Line 3: C        5.03367812       6.44217175       3.36017192       1.27264249      -0.46241006      -1.00416665

=== First Pure MASnI3 Structure ===
Structure index: 1
Formula: C8H48I24N8Sn8
Atoms: 96

=== Available Metadata ===
filename: empty-filename

=== Available Arrays ===
numbers Shape: (96,)
positions Shape: (96, 3)

=== DFT Label Availability ===
Calculator: SinglePointCalculator(energy=0.32516686827875674, forces=..., free_energy=-6102514.98896051)
energy: info=False, arrays=False, calculator=True
forces: info=False, arrays=False, calculator=True
stress: info=False, arrays=False, calculator=False
virial: info=False, arrays=False, calculator=False

Label inspection completed!


確認170筆中是否可作為Fine Tuning的資料來源

In [15]:
# =====================================================
# Step 39C: MASnI3 DFT Energy-Force Audit
# =====================================================

from ase.io import iread
from collections import Counter
import numpy as np
import pandas as pd

file_path = (
    "/content/drive/MyDrive/MASnI3_DFT_Dataset/"
    "External_DFT/MA_Ge_Sn_I3_train.xyz"
)

results = []

for i, atoms in enumerate(iread(file_path, index=":")):

    counts = Counter(atoms.get_chemical_symbols())
    n = counts.get("Sn", 0)

    is_masni3 = (
        n > 0
        and set(counts) == {"C", "H", "N", "Sn", "I"}
        and counts["C"] == n
        and counts["H"] == 6 * n
        and counts["N"] == n
        and counts["I"] == 3 * n
    )

    if not is_masni3:
        continue

    calc = atoms.calc

    if calc is None:
        continue

    energy = calc.results.get("energy")
    forces = calc.results.get("forces")
    free_energy = calc.results.get("free_energy")

    energy_value = (
        float(energy)
        if energy is not None
        else np.nan
    )

    forces_array = (
        np.asarray(forces, dtype=float)
        if forces is not None
        else None
    )

    max_force = (
        float(np.linalg.norm(forces_array, axis=1).max())
        if forces_array is not None
        else np.nan
    )

    results.append({
        "index": i,
        "n_atoms": len(atoms),
        "energy": energy_value,
        "energy_per_atom": energy_value / len(atoms),
        "free_energy": (
            float(free_energy)
            if free_energy is not None
            else np.nan
        ),
        "max_force": max_force,
        "has_forces": forces_array is not None,
        "has_stress": "stress" in calc.results
    })

audit_df = pd.DataFrame(results)

print("=== MASnI3 DFT Energy-Force Audit ===")

print("Pure MASnI3 configurations:", len(audit_df))

print("\n=== Label Availability ===")
print("Energy available:", audit_df["energy"].notna().sum())
print("Forces available:", audit_df["has_forces"].sum())
print("Stress available:", audit_df["has_stress"].sum())

print("\n=== Energy per Atom Distribution ===")
print(audit_df["energy_per_atom"].describe())

print("\n=== Free Energy Distribution ===")
print(audit_df["free_energy"].describe())

print("\n=== Maximum Force Distribution ===")
print(audit_df["max_force"].describe())

print("\n=== Structures by Atom Count ===")
print(audit_df["n_atoms"].value_counts().sort_index())

print("\n=== Finite Value Verification ===")

valid_energy = np.isfinite(
    audit_df["energy_per_atom"]
).sum()

valid_force = np.isfinite(
    audit_df["max_force"]
).sum()

print("Finite energies:", valid_energy)
print("Finite maximum forces:", valid_force)

print("\nEnergy-force audit completed!")

=== MASnI3 DFT Energy-Force Audit ===
Pure MASnI3 configurations: 170

=== Label Availability ===
Energy available: 170
Forces available: 170
Stress available: 4

=== Energy per Atom Distribution ===
count    170.000000
mean      -0.108435
std        0.158507
min       -0.184125
25%       -0.176659
50%       -0.157153
75%       -0.107828
max        1.147264
Name: energy_per_atom, dtype: float64

=== Free Energy Distribution ===
count    1.700000e+02
mean    -5.932014e+06
std      1.429513e+06
min     -1.220506e+07
25%     -6.102532e+06
50%     -6.102530e+06
75%     -6.102524e+06
max     -7.628164e+05
Name: free_energy, dtype: float64

=== Maximum Force Distribution ===
count    170.000000
mean       3.172621
std        4.692358
min        0.069432
25%        0.635374
50%        1.726231
75%        4.131191
max       36.570596
Name: max_force, dtype: float64

=== Structures by Atom Count ===
n_atoms
12       2
24       2
48      11
96     151
192      4
Name: count, dtype: int64

=== Fi